<a href="https://colab.research.google.com/github/armin-lawi/ForcastingStockPrice-with-Grouped-Dataset/blob/main/GRU/GroupPredict_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Preparation Data**

In [ ]:
# Install required Libraries
!pip install pandas_datareader

In [ ]:
# Install required Libraries
!pip install yfinance --upgrade --no-cache-dir

In [ ]:
## Import Libraries and set information
from pandas_datareader import data as pdr
from datetime import date

import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from numpy import array
import math

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, concatenate, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.backend import square, mean
from tensorflow.keras.utils import plot_model
import yfinance as yf

# Get Current Date
today = date.today()
currentDate = today.strftime("%Y-%m-%d")

# Set Info
start_date = '2010-01-04'
end_date = "2025-09-09" #currentDate
stockName = ['AMZN','GOOGL','BALL','QCOM']
np.set_printoptions(threshold=sys.maxsize)

In [ ]:
# Data AMZN
stock_amzn = yf.download(stockName[0], start_date, end_date)
stock_amzn

In [ ]:
# Data GOOGL
stock_googl = yf.download(stockName[1], start_date, end_date)
stock_googl

In [ ]:
# Data BLL
stock_bll = yf.download(stockName[2], start_date, end_date)
stock_bll

In [ ]:
# Data QCOM
stock_qcom = yf.download(stockName[3], start_date, end_date)
stock_qcom

In [ ]:
# Visualization data

#close
plt.figure(figsize=(16,8))
plt.title('Close Price')
plt.plot(stock_amzn['Close'])
plt.plot(stock_googl['Close'])

plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.legend(['AMZN','GOOGL'])
plt.grid()

currentFig = plt.gcf()
currentFig.set_facecolor('white')
plt.show()

#close
plt.figure(figsize=(16,8))
plt.title('Close Price')
plt.plot(stock_bll['Close'])
plt.plot(stock_qcom['Close'])

plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.legend(['BLL','QCOM'])
plt.grid()

currentFig = plt.gcf()
currentFig.set_facecolor('white')
plt.show()

In [ ]:
# Extract Closing price
data_amzn = stock_amzn.loc[:, ('Close', 'AMZN')].to_frame()
dataset_amzn = data_amzn.values
# Extract Closing price
data_googl = stock_googl.loc[:, ('Close', 'GOOGL')].to_frame()
dataset_googl = data_googl.values
# Extract Closing price
data_bll = stock_bll.loc[:, ('Close', 'BALL')].to_frame()
dataset_bll = data_bll.values
# Extract Closing price
data_qcom = stock_qcom.loc[:, ('Close', 'QCOM')].to_frame()
dataset_qcom = data_qcom.values

In [ ]:
# Rename column
data_amzn.rename(columns={'Close': 'Close_amzn'}, inplace=True)
data_googl.rename(columns={'Close': 'Close_googl'}, inplace=True)
data_bll.rename(columns={'Close': 'Close_bll'}, inplace=True)
data_qcom.rename(columns={'Close': 'Close_qcom'}, inplace=True)

In [ ]:
# Preprocess the data AMZN
normalizer = MinMaxScaler(feature_range=(0,1)) # instantiate scaler
normalizedData_amzn = normalizer.fit_transform(dataset_amzn) # values between 0,1
print(normalizedData_amzn)

In [ ]:
# Storing the number of data points in the array
num_data_amzn = len(normalizedData_amzn)
num_days_used = 60
data_used_amzn = np.array([normalizedData_amzn[i : i + num_days_used].copy() for i in range(num_data_amzn - num_days_used)])

data_to_predict_amzn = np.array(normalizedData_amzn[(num_days_used):, :1])

# Creating a dates array for the dates that were used by data_used
dates_used_amzn = stock_amzn.index[num_days_used:num_data_amzn]

# Storing the scaler object for prediction later
y_normaliser_amzn = MinMaxScaler()
y_normaliser_amzn.fit(data_amzn[['Close_amzn']].to_numpy()[num_days_used:])

display(normalizedData_amzn.shape, data_used_amzn.shape,data_to_predict_amzn.shape, dates_used_amzn.shape)

In [ ]:
train_split = 0.8
data_size = data_used_amzn.shape[0]
num_features_amzn = data_used_amzn.shape[2]
train_size_amzn = int(data_size * train_split)
test_size_amzn = data_size - train_size_amzn

# Splitting the dataset up into train and test sets
X_train_amzn = data_used_amzn[0:train_size_amzn, :, :]
y_train_amzn = data_to_predict_amzn[0:train_size_amzn, :]
dates_train_amzn = dates_used_amzn[0:train_size_amzn]
X_test_amzn = data_used_amzn[train_size_amzn:, :, :]
y_test_amzn = data_to_predict_amzn[train_size_amzn:, :]
dates_test_amzn = dates_used_amzn[train_size_amzn:]

unscaled_y_train_amzn = data_amzn['Close_amzn'].to_numpy()[(num_days_used):][0:train_size_amzn]
unscaled_y_test_amzn = data_amzn['Close_amzn'].to_numpy()[(num_days_used):][train_size_amzn:]

display("X_train shape:", X_train_amzn.shape, "X_test shape:", X_test_amzn.shape, "y_train shape:", y_train_amzn.shape, "y_test shape:", y_test_amzn.shape)

In [ ]:
# Preprocess the data GOOGL
normalizer = MinMaxScaler(feature_range=(0,1)) # instantiate scaler
normalizedData_googl = normalizer.fit_transform(dataset_googl) # values between 0,1
print(normalizedData_googl)

In [ ]:
# Storing the number of data points in the array
num_data_googl = len(normalizedData_googl)
num_days_used = 60
data_used_googl = np.array([normalizedData_googl[i : i + num_days_used].copy() for i in range(num_data_googl - num_days_used)])

data_to_predict_googl = np.array(normalizedData_googl[(num_days_used):, :1])

# Creating a dates array for the dates that were used by data_used
dates_used_googl = stock_googl.index[num_days_used:num_data_googl]

# Storing the scaler object for prediction later
y_normaliser_googl = MinMaxScaler()
y_normaliser_googl.fit(data_googl[['Close_googl']].to_numpy()[num_days_used:])
num_features_googl = data_used_googl.shape[2]

display(normalizedData_googl.shape, data_used_googl.shape,data_to_predict_googl.shape, dates_used_googl.shape)

In [ ]:
train_split = 0.8
data_size = data_used_googl.shape[0]
num_features_googl = data_used_googl.shape[2]
train_size_googl = int(data_size * train_split)
test_size_googl = data_size - train_size_googl

# Splitting the dataset up into train and test sets
X_train_googl = data_used_googl[0:train_size_googl, :, :]
y_train_googl = data_to_predict_googl[0:train_size_amzn, :]
dates_train_googl = dates_used_googl[0:train_size_googl]
X_test_googl = data_used_googl[train_size_googl:, :, :]
y_test_googl = data_to_predict_googl[train_size_googl:, :]
dates_test_googl = dates_used_googl[train_size_googl:]

unscaled_y_train_googl = data_googl['Close_googl'].to_numpy()[(num_days_used):][0:train_size_googl]
unscaled_y_test_googl = data_googl['Close_googl'].to_numpy()[(num_days_used):][train_size_googl:]

display("X_train shape:", X_train_googl.shape, "X_test shape:", X_test_googl.shape, "y_train shape:", y_train_googl.shape, "y_test shape:", y_test_googl.shape)

In [ ]:
# Preprocess the data BLL
normalizer = MinMaxScaler(feature_range=(0,1)) # instantiate scaler
normalizedData_bll = normalizer.fit_transform(dataset_bll) # values between 0,1
print(normalizedData_bll)

In [ ]:
# Storing the number of data points in the array
num_data_bll = len(normalizedData_bll)
num_days_used = 60
data_used_bll = np.array([normalizedData_bll[i : i + num_days_used].copy() for i in range(num_data_bll - num_days_used)])

data_to_predict_bll = np.array(normalizedData_bll[(num_days_used):, :1])

# Creating a dates array for the dates that were used by data_used
dates_used_bll = stock_bll.index[num_days_used:num_data_bll]

# Storing the scaler object for prediction later
y_normaliser_bll = MinMaxScaler()
y_normaliser_bll.fit(data_bll[['Close_bll']].to_numpy()[num_days_used:])
num_features_bll = data_used_bll.shape[2]

display(normalizedData_bll.shape, data_used_bll.shape,data_to_predict_bll.shape, dates_used_bll.shape)

In [ ]:
train_split = 0.8
data_size = data_used_bll.shape[0]
num_features_bll = data_used_bll.shape[2]
train_size_bll = int(data_size * train_split)
test_size_bll = data_size - train_size_bll

# Splitting the dataset up into train and test sets
X_train_bll = data_used_bll[0:train_size_bll, :, :]
y_train_bll = data_to_predict_bll[0:train_size_bll, :]
dates_train_bll = dates_used_bll[0:train_size_bll]
X_test_bll = data_used_bll[train_size_bll:, :, :]
y_test_bll = data_to_predict_bll[train_size_bll:, :]
dates_test_bll = dates_used_bll[train_size_bll:]

unscaled_y_train_bll = data_bll['Close_bll'].to_numpy()[(num_days_used):][0:train_size_bll]
unscaled_y_test_bll = data_bll['Close_bll'].to_numpy()[(num_days_used):][train_size_bll:]

display("X_train shape:", X_train_bll.shape, "X_test shape:", X_test_bll.shape, "y_train shape:", y_train_bll.shape, "y_test shape:", y_test_bll.shape)

In [ ]:
# Preprocess the data QCOM
normalizer = MinMaxScaler(feature_range=(0,1)) # instantiate scaler
normalizedData_qcom = normalizer.fit_transform(dataset_qcom) # values between 0,1
print(normalizedData_qcom)

In [ ]:
# Storing the number of data points in the array
num_data_qcom = len(normalizedData_qcom)
num_days_used = 60
data_used_qcom = np.array([normalizedData_qcom[i : i + num_days_used].copy() for i in range(num_data_qcom - num_days_used)])

data_to_predict_qcom = np.array(normalizedData_qcom[(num_days_used):, :1])

# Creating a dates array for the dates that were used by data_used
dates_used_qcom = stock_qcom.index[num_days_used:num_data_qcom]

# Storing the scaler object for prediction later
y_normaliser_qcom = MinMaxScaler()
y_normaliser_qcom.fit(data_qcom[['Close_qcom']].to_numpy()[num_days_used:])
num_features_qcom = data_used_qcom.shape[2]

display(normalizedData_qcom.shape, data_used_qcom.shape,data_to_predict_qcom.shape, dates_used_qcom.shape)

In [ ]:
train_split = 0.8
data_size = data_used_qcom.shape[0]
num_features_qcom = data_used_qcom.shape[2]
train_size_qcom = int(data_size * train_split)
test_size_qcom = data_size - train_size_qcom

# Splitting the dataset up into train and test sets
X_train_qcom = data_used_qcom[0:train_size_qcom, :, :]
y_train_qcom = data_to_predict_qcom[0:train_size_qcom, :]
dates_train_qcom = dates_used_qcom[0:train_size_qcom]
X_test_qcom = data_used_qcom[train_size_qcom:, :, :]
y_test_qcom = data_to_predict_qcom[train_size_qcom:, :]
dates_test_qcom = dates_used_qcom[train_size_qcom:]

unscaled_y_train_qcom = data_qcom['Close_qcom'].to_numpy()[(num_days_used):][0:train_size_qcom]
unscaled_y_test_qcom = data_qcom['Close_qcom'].to_numpy()[(num_days_used):][train_size_qcom:]

display("X_train shape:", X_train_qcom.shape, "X_test shape:", X_test_qcom.shape, "y_train shape:", y_train_qcom.shape, "y_test shape:", y_test_qcom.shape)

In [ ]:
# Concat the 4 stocks data
stock_df = pd.DataFrame()
stock_df = pd.concat([stock_df, data_amzn, data_googl, data_bll, data_qcom], axis=1)

In [ ]:
stock_df

In [ ]:
data = stock_df.loc[:, ['Close_amzn', 'Close_googl', 'Close_bll', 'Close_qcom']]
dataset = data.values


In [ ]:
dataset

In [ ]:
# Preprocess the data
normalizer = MinMaxScaler(feature_range=(0,1)) # instantiate scaler
normalizedData = normalizer.fit_transform(dataset) # values between 0,1
print(normalizedData)

In [ ]:
# Visualization Data
#close
plt.figure(figsize=(16,8))
plt.title('Close Price')
plt.plot(normalizedData[:,:4])

plt.xlabel('Date')
plt.ylabel('Stock Price')
plt.legend(['AMZN','GOOGL','BLL','QCOM'])
plt.grid()

currentFig = plt.gcf()
currentFig.set_facecolor('white')
plt.show()

In [ ]:
# Storing the number of data points in the array
num_data = len(normalizedData)
num_days_used = 60
data_used = np.array([normalizedData[i : i + num_days_used].copy() for i in range(num_data - num_days_used)])

data_to_predict = np.array(normalizedData[(num_days_used):, :6])

# Creating a dates array for the dates that were used by data_used
dates_used = stock_df.index[num_days_used:num_data]

# Storing the scaler object for prediction later
y_normaliser = MinMaxScaler()
y_normaliser.fit(stock_df[['Close_amzn','Close_googl','Close_bll','Close_qcom']].to_numpy()[num_days_used:])

display(normalizedData.shape, data_used.shape,data_to_predict.shape, dates_used.shape)

In [ ]:

train_split = 0.8
data_size = data_used.shape[0]
num_features = data_used.shape[2]
train_size = int(data_size * train_split)
test_size = data_size - train_size

# Splitting the dataset up into train and test sets
X_train = data_used[0:train_size, :, :]
y_train = data_to_predict[0:train_size, :]
dates_train = dates_used[0:train_size]
X_test = data_used[train_size:, :, :]
y_test = data_to_predict[train_size:, :]
dates_test = dates_used[train_size:]

unscaled_y_train = stock_df[['Close_amzn','Close_googl','Close_bll','Close_qcom']].to_numpy()[(num_days_used):][0:train_size, :]
unscaled_y_test = stock_df[['Close_amzn','Close_googl','Close_bll','Close_qcom']].to_numpy()[(num_days_used):][train_size:, :]

display("X_train shape:", X_train.shape, "y_train shape:", y_train.shape,
        "X_test shape:", X_test.shape, "y_test shape:", y_test.shape,
        "unscaled_y_train shape:", unscaled_y_train.shape, "unscaled_y_test shape:", unscaled_y_test.shape)

In [ ]:

# Save data to excel
data_ = pd.DataFrame(data=data)
normalizedData_ = pd.DataFrame(data=normalizedData)

file_name1 = 'RealData.xlsx'
file_name2 = 'RealNormData.xlsx'

data_.to_excel(file_name1)
normalizedData_.to_excel(file_name2)

#Attention Mechanism

In [ ]:
from tensorflow.keras.layers import Layer
import tensorflow.keras.backend as K

class AttentionLayer(Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="random_normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        # inputs.shape = (batch_size, time_steps, input_dim)
        e = K.tanh(K.dot(inputs, self.W) + self.b)
        a = K.softmax(e, axis=1)
        output = inputs * a
        return K.sum(output, axis=1)

# **Model1**

In [ ]:
input_amzn = Input(shape=(num_days_used, num_features_amzn), name = 'input_amzn')
input_googl = Input(shape=(num_days_used, num_features_googl), name = 'input_googl')
input_bll = Input(shape=(num_days_used, num_features_bll), name = 'input_bll')
input_qcom = Input(shape=(num_days_used, num_features_qcom), name = 'input_qcom')

# LSTM branch for efficient initial processing
x1 = GRU(160, return_sequences=True, name='gru_amzn')(input_amzn)
x1 = Dropout(0.5)(x1)

# GRU branch to capture long-term dependencies in parallel
x1 = LSTM(160, return_sequences=True, name='lstm_amzn')(x1)
x1 = Dropout(0.5)(x1)

# LSTM branch for efficient initial processing
x2 = GRU(160, return_sequences=True, name='gru_googl')(input_googl)
x2 = Dropout(0.5)(x2)

# GRU branch to capture long-term dependencies in parallel
x2 = LSTM(160, return_sequences=True, name='lstm_googl')(x2)
x2 = Dropout(0.5)(x2)

# LSTM branch for efficient initial processing
x3 = GRU(160, return_sequences=True, name='gru_bll')(input_bll)
x3 = Dropout(0.5)(x3)

# GRU branch to capture long-term dependencies in parallel
x3 = LSTM(160, return_sequences=True, name='lstm_bll')(x3)
x3 = Dropout(0.5)(x3)

# LSTM branch for efficient initial processing
x4 = GRU(160, return_sequences=True, name='gru_qcom')(input_qcom)
x4 = Dropout(0.5)(x4)

# GRU branch to capture long-term dependencies in parallel
x4 = LSTM(160, return_sequences=True, name='lstm_qcom')(x4)
x4 = Dropout(0.5)(x4)


conc = concatenate([x1,x2,x3,x4])

output1 = LSTM(160, name='amzn_0')(conc)
output1 = Dense(1, name='amzn_final')(output1)

output2 = LSTM(160, name='googl_0')(conc)
output2 = Dense(1, name='googl_final')(output2)

output3 = LSTM(160, name='bll_0')(conc)
output3 = Dense(1, name='bll_final')(output3)

output4 = LSTM(160, name='qcom_0')(conc)
output4 = Dense(1, name='qcom_final')(output4)

model3 = Model(inputs = [input_amzn, input_googl, input_bll, input_qcom], outputs = [output1, output2, output3, output4])

adam = Adam(learning_rate=0.001)

model3.compile(optimizer=adam, loss='mse')
model3.summary()

In [ ]:
# Displaying the structure of the final model
plot_model(model3, show_shapes=True)

In [ ]:
# Fitting Model
history = model3.fit(x=[X_train_amzn,X_train_googl,X_train_bll,X_train_qcom], y=[y_train_amzn,y_train_googl,y_train_bll,y_train_qcom], batch_size=32, epochs=30, validation_split=0.2)
evaluation = model3.evaluate([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom], [y_test_amzn,y_test_googl,y_test_bll,y_test_qcom])
print(evaluation)

In [ ]:
# Prediction data test
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model3.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
preds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_test_pred = preds_arr

amzn=0
googl=1
bll=2
qcom=3

plt.gcf().set_size_inches(22, 15, forward=True)
currentFig.set_facecolor('white')

real = plt.plot(y_test[:,:], label='real')
pred = plt.plot(y_test_pred[:,:], label='predicted')

plt.legend(['real amzn','real googl','real bll','real qcom','predict amzn','predic googl','predict bll','predict qcom'])
plt.xlabel('Days being predicted (units are arbitrary)', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close Price on the Test Set', fontsize=30)

plt.show()

In [ ]:
# Prediction data of each company
y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred= model3.predict([X_train_amzn,X_train_googl,X_train_bll,X_train_qcom])
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model3.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
trainpreds_arr = np.hstack((y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred))
testpreds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_train_pred = y_normaliser.inverse_transform(trainpreds_arr)
y_test_pred = y_normaliser.inverse_transform(testpreds_arr)

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_amzn'], label='real amzn price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,amzn], label='predicted train amzn', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,amzn], label='predicted test amzn', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close AMZN Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_googl'], label='real googl price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,googl], label='predicted train googl', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,googl], label='predicted test googl', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close GOOGL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_bll'], label='real bll price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,bll], label='predicted train bll', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,bll], label='predicted test bll', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close BLL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_qcom'], label='real qcom price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,qcom], label='predicted train qcom', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,qcom], label='predicted test qcom', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close QCOM Price on the Train and Test Set', fontsize=30)
plt.show()

In [ ]:
# Visualization data
loss = history.history['loss']
val_loss = history.history['val_loss']
amzn_loss = history.history['amzn_final_loss']
val_amzn_loss = history.history['val_amzn_final_loss']
googl_loss = history.history['googl_final_loss']
val_googl_loss = history.history['val_googl_final_loss']
bll_loss = history.history['bll_final_loss']
val_bll_loss = history.history['val_bll_final_loss']
qcom_loss = history.history['qcom_final_loss']
val_qcom_loss = history.history['val_qcom_final_loss']
epochs = range(1, len(loss) + 1)
plt.figure()

#Train and validation loss
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()
plt.show()

plt.plot(epochs, amzn_loss, 'b', label='Training loss')
plt.plot(epochs, val_amzn_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of AMZN')
plt.legend()
plt.show()

plt.plot(epochs, googl_loss, 'b', label='Training loss')
plt.plot(epochs, val_googl_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of GOOGL')
plt.legend()
plt.show()

plt.plot(epochs, bll_loss, 'b', label='Training loss')
plt.plot(epochs, val_bll_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of BLL')
plt.legend()
plt.show()

plt.plot(epochs, qcom_loss, 'b', label='Training loss')
plt.plot(epochs, val_qcom_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of QCOM')
plt.legend()
plt.show()

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of AMZN Price

# Calculating MAE performance metrics
mae_amzn_train = mean_absolute_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MAE
mae_amzn_test = mean_absolute_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating MSE performance metrics
mse_amzn_train = mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MSE
mse_amzn_test = mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating RMSE performance metrics
rmse_amzn_train = math.sqrt(mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn]))

# Calculating Test Data RMSE
rmse_amzn_test = math.sqrt(mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn]))

# Calculating MAPE performance metrics
mape_amzn_train = np.mean(np.abs((unscaled_y_train_amzn - y_train_pred[:,amzn])/unscaled_y_train_amzn))*100

# Calculating Test Data MAPE
mape_amzn_test = np.mean(np.abs((unscaled_y_test_amzn - y_test_pred[:,amzn])/unscaled_y_test_amzn))*100

# Calculating R² performance metrics
r2_amzn_train = r2_score(unscaled_y_train_amzn, y_train_pred[:, amzn])

# Calculating Test Data R²
r2_amzn_test = r2_score(unscaled_y_test_amzn, y_test_pred[:, amzn])

print('Evaluation of AMZN price','\nMAE Train:', mae_amzn_train, '\nMAE Test:', mae_amzn_test,
      '\nMSE Train:', mse_amzn_train, '\nMSE Test:', mse_amzn_test,
      '\nRMSE Train1:', rmse_amzn_train, '\nRMSE Test1:', rmse_amzn_test,
      '\nMAPE Train:', mape_amzn_train, '\nMAPE Test:', mape_amzn_test,
      '\nR² Train:', r2_amzn_train, '\nR² Test:', r2_amzn_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE, r2 of GOOGL Price

# Calculating MAE performance metrics
mae_googl_train = mean_absolute_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MAE
mae_googl_test = mean_absolute_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating MSE performance metrics
mse_googl_train = mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MSE
mse_googl_test = mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating RMSE performance metrics
rmse_googl_train = math.sqrt(mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl]))

# Calculating Test Data RMSE
rmse_googl_test = math.sqrt(mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl]))

# Calculating MAPE performance metrics
mape_googl_train = np.mean(np.abs((unscaled_y_train_googl - y_train_pred[:,googl])/unscaled_y_train_googl))*100

# Calculating Test Data MAPE
mape_googl_test = np.mean(np.abs((unscaled_y_test_googl - y_test_pred[:,googl])/unscaled_y_test_googl))*100

# Calculating R² performance metrics
r2_googl_train = r2_score(unscaled_y_train_googl, y_train_pred[:, googl])

# Calculating Test Data R²
r2_googl_test = r2_score(unscaled_y_test_googl, y_test_pred[:, googl])

print('Evaluation of GOOGL price','\nMAE Train:', mae_googl_train, '\nMAE Test:', mae_googl_test,
      '\nMSE Train:', mse_googl_train, '\nMSE Test:', mse_googl_test,
      '\nRMSE Train1:', rmse_googl_train, '\nRMSE Test1:', rmse_googl_test,
      '\nMAPE Train:', mape_googl_train, '\nMAPE Test:', mape_googl_test,
      '\nR² Train:', r2_googl_train, '\nR² Test:', r2_googl_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of BLL Price

# Calculating MAE performance metrics
mae_bll_train = mean_absolute_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MAE
mae_bll_test = mean_absolute_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating MSE performance metrics
mse_bll_train = mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MSE
mse_bll_test = mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating RMSE performance metrics
rmse_bll_train = math.sqrt(mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll]))

# Calculating Test Data RMSE
rmse_bll_test = math.sqrt(mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll]))

# Calculating MAPE performance metrics
mape_bll_train = np.mean(np.abs((unscaled_y_train_bll - y_train_pred[:,bll])/unscaled_y_train_bll))*100

# Calculating Test Data MAPE
mape_bll_test = np.mean(np.abs((unscaled_y_test_bll - y_test_pred[:,bll])/unscaled_y_test_bll))*100

# Calculating R² performance metrics
r2_bll_train = r2_score(unscaled_y_train_bll, y_train_pred[:, bll])

# Calculating Test Data R²
r2_bll_test = r2_score(unscaled_y_test_bll, y_test_pred[:, bll])

print('Evaluation of BLL price','\nMAE Train:', mae_bll_train, '\nMAE Test:', mae_bll_test,
      '\nMSE Train:', mse_bll_train, '\nMSE Test:', mse_bll_test,
      '\nRMSE Train1:', rmse_bll_train, '\nRMSE Test1:', rmse_bll_test,
      '\nMAPE Train:', mape_bll_train, '\nMAPE Test:', mape_bll_test,
      '\nR² Train:', r2_bll_train, '\nR² Test:', r2_bll_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of QCOM Price

# Calculating MAE performance metrics
mae_qcom_train = mean_absolute_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MAE
mae_qcom_test = mean_absolute_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating MSE performance metrics
mse_qcom_train = mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MSE
mse_qcom_test = mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating RMSE performance metrics
rmse_qcom_train = math.sqrt(mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom]))

# Calculating Test Data RMSE
rmse_qcom_test = math.sqrt(mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom]))

# Calculating MAPE performance metrics
mape_qcom_train = np.mean(np.abs((unscaled_y_train_qcom - y_train_pred[:,qcom])/unscaled_y_train_qcom))*100

# Calculating Test Data MAPE
mape_qcom_test = np.mean(np.abs((unscaled_y_test_qcom - y_test_pred[:,qcom])/unscaled_y_test_qcom))*100

# Calculating R² performance metrics
r2_qcom_train = r2_score(unscaled_y_train_qcom, y_train_pred[:, qcom])

# Calculating Test Data R²
r2_qcom_test = r2_score(unscaled_y_test_qcom, y_test_pred[:, qcom])

print('Evaluation of QCOM price','\nMAE Train:', mae_qcom_train, '\nMAE Test:', mae_qcom_test,
      '\nMSE Train:', mse_qcom_train, '\nMSE Test:', mse_qcom_test,
      '\nRMSE Train1:', rmse_qcom_train, '\nRMSE Test1:', rmse_qcom_test,
      '\nMAPE Train:', mape_qcom_train, '\nMAPE Test:', mape_qcom_test,
      '\nR² Train:', r2_qcom_train, '\nR² Test:', r2_qcom_test)

In [ ]:
# Save data to excel
df_pred_test_norm = pd.DataFrame(data=testpreds_arr)
df_pred_train_norm = pd.DataFrame(data=trainpreds_arr)
df_pred_test = pd.DataFrame(data=y_test_pred)
df_pred_train = pd.DataFrame(data=y_train_pred)

file_name11 = 'TestPredNormData3.xlsx'
file_name12 = 'TrainPredNormData3.xlsx'
file_name13 = 'TestPredData3.xlsx'
file_name14 = 'TrainPredData3.xlsx'

df_pred_test_norm.to_excel(file_name11)
df_pred_train_norm.to_excel(file_name12)
df_pred_test.to_excel(file_name13)
df_pred_train.to_excel(file_name14)

# **Model2**

In [ ]:
input_amzn = Input(shape=(num_days_used, num_features_amzn), name = 'input_amzn')
input_googl = Input(shape=(num_days_used, num_features_googl), name = 'input_googl')
input_bll = Input(shape=(num_days_used, num_features_bll), name = 'input_bll')
input_qcom = Input(shape=(num_days_used, num_features_qcom), name = 'input_qcom')

# LSTM branch for efficient initial processing
x1 = GRU(160, return_sequences=True, name='gru_amzn')(input_amzn)
x1 = Dropout(0.5)(x1)

# GRU branch to capture long-term dependencies in parallel
x1 = LSTM(160, return_sequences=True, name='lstm_amzn')(x1)
x1 = Dropout(0.5)(x1)

# LSTM branch for efficient initial processing
x2 = GRU(160, return_sequences=True, name='gru_googl')(input_googl)
x2 = Dropout(0.5)(x2)

# GRU branch to capture long-term dependencies in parallel
x2 = LSTM(160, return_sequences=True, name='lstm_googl')(x2)
x2 = Dropout(0.5)(x2)

# LSTM branch for efficient initial processing
x3 = GRU(160, return_sequences=True, name='gru_bll')(input_bll)
x3 = Dropout(0.5)(x3)

# GRU branch to capture long-term dependencies in parallel
x3 = LSTM(160, return_sequences=True, name='lstm_bll')(x3)
x3 = Dropout(0.5)(x3)

# LSTM branch for efficient initial processing
x4 = GRU(160, return_sequences=True, name='gru_qcom')(input_qcom)
x4 = Dropout(0.5)(x4)

# GRU branch to capture long-term dependencies in parallel
x4 = LSTM(160, return_sequences=True, name='lstm_qcom')(x4)
x4 = Dropout(0.5)(x4)

conc = concatenate([x1,x2,x3,x4])
conc = LSTM(160, return_sequences=True, name='lstm_conc')(conc)

output1 = GRU(160, name='amzn_0')(conc)
output1 = Dense(1, name='amzn_final')(output1)

output2 = GRU(160, name='googl_0')(conc)
output2 = Dense(1, name='googl_final')(output2)

output3 = GRU(160, name='bll_0')(conc)
output3 = Dense(1, name='bll_final')(output3)

output4 = GRU(160, name='qcom_0')(conc)
output4 = Dense(1, name='qcom_final')(output4)


model4 = Model(inputs = [input_amzn, input_googl, input_bll, input_qcom], outputs = [output1, output2, output3, output4])

adam = Adam(learning_rate=0.001)

model4.compile(optimizer=adam, loss='mse')
model4.summary()

In [ ]:
# Displaying the structure of the final model
plot_model(model4, show_shapes=True)

In [ ]:
# fitting Model
history = model4.fit(x=[X_train_amzn,X_train_googl,X_train_bll,X_train_qcom], y=[y_train_amzn,y_train_googl,y_train_bll,y_train_qcom], batch_size=32, epochs=30, validation_split=0.2)
evaluation = model4.evaluate([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom], [y_test_amzn,y_test_googl,y_test_bll,y_test_qcom])
print(evaluation)

In [ ]:
# Prediction data test
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model4.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
preds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_test_pred = preds_arr

amzn=0
googl=1
bll=2
qcom=3

plt.gcf().set_size_inches(22, 15, forward=True)
currentFig.set_facecolor('white')

real = plt.plot(y_test[:,:], label='real')
pred = plt.plot(y_test_pred[:,:], label='predicted')

plt.legend(['real amzn','real googl','real bll','real qcom','predict amzn','predic googl','predict bll','predict qcom'])
plt.xlabel('Days being predicted (units are arbitrary)', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close Price on the Test Set', fontsize=30)

plt.show()

In [ ]:
# Prediction data of each company
y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred= model4.predict([X_train_amzn,X_train_googl,X_train_bll,X_train_qcom])
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model4.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
trainpreds_arr = np.hstack((y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred))
testpreds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_train_pred = y_normaliser.inverse_transform(trainpreds_arr)
y_test_pred = y_normaliser.inverse_transform(testpreds_arr)

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_amzn'], label='real amzn price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,amzn], label='predicted train amzn', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,amzn], label='predicted test amzn', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close AMZN Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_googl'], label='real googl price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,googl], label='predicted train googl', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,googl], label='predicted test googl', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close GOOGL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_bll'], label='real bll price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,bll], label='predicted train bll', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,bll], label='predicted test bll', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close BLL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_qcom'], label='real qcom price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,qcom], label='predicted train qcom', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,qcom], label='predicted test qcom', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close QCOM Price on the Train and Test Set', fontsize=30)
plt.show()

In [ ]:
# visualization Loss
loss = history.history['loss']
val_loss = history.history['val_loss']
amzn_loss = history.history['amzn_final_loss']
val_amzn_loss = history.history['val_amzn_final_loss']
googl_loss = history.history['googl_final_loss']
val_googl_loss = history.history['val_googl_final_loss']
bll_loss = history.history['bll_final_loss']
val_bll_loss = history.history['val_bll_final_loss']
qcom_loss = history.history['qcom_final_loss']
val_qcom_loss = history.history['val_qcom_final_loss']
epochs = range(1, len(loss) + 1)
plt.figure()

#Train and validation loss
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()
plt.show()

plt.plot(epochs, amzn_loss, 'b', label='Training loss')
plt.plot(epochs, val_amzn_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of AMZN')
plt.legend()
plt.show()

plt.plot(epochs, googl_loss, 'b', label='Training loss')
plt.plot(epochs, val_googl_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of GOOGL')
plt.legend()
plt.show()

plt.plot(epochs, bll_loss, 'b', label='Training loss')
plt.plot(epochs, val_bll_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of BLL')
plt.legend()
plt.show()

plt.plot(epochs, qcom_loss, 'b', label='Training loss')
plt.plot(epochs, val_qcom_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of QCOM')
plt.legend()
plt.show()

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of AMZN Price

# Calculating MAE performance metrics
mae_amzn_train = mean_absolute_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MAE
mae_amzn_test = mean_absolute_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating MSE performance metrics
mse_amzn_train = mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MSE
mse_amzn_test = mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating RMSE performance metrics
rmse_amzn_train = math.sqrt(mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn]))

# Calculating Test Data RMSE
rmse_amzn_test = math.sqrt(mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn]))

# Calculating MAPE performance metrics
mape_amzn_train = np.mean(np.abs((unscaled_y_train_amzn - y_train_pred[:,amzn])/unscaled_y_train_amzn))*100

# Calculating Test Data MAPE
mape_amzn_test = np.mean(np.abs((unscaled_y_test_amzn - y_test_pred[:,amzn])/unscaled_y_test_amzn))*100

# Calculating R² performance metrics
r2_amzn_train = r2_score(unscaled_y_train_amzn, y_train_pred[:, amzn])

# Calculating Test Data R²
r2_amzn_test = r2_score(unscaled_y_test_amzn, y_test_pred[:, amzn])

print('Evaluation of AMZN price','\nMAE Train:', mae_amzn_train, '\nMAE Test:', mae_amzn_test,
      '\nMSE Train:', mse_amzn_train, '\nMSE Test:', mse_amzn_test,
      '\nRMSE Train1:', rmse_amzn_train, '\nRMSE Test1:', rmse_amzn_test,
      '\nMAPE Train:', mape_amzn_train, '\nMAPE Test:', mape_amzn_test,
      '\nR² Train:', r2_amzn_train, '\nR² Test:', r2_amzn_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE, r2 of GOOGL Price

# Calculating MAE performance metrics
mae_googl_train = mean_absolute_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MAE
mae_googl_test = mean_absolute_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating MSE performance metrics
mse_googl_train = mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MSE
mse_googl_test = mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating RMSE performance metrics
rmse_googl_train = math.sqrt(mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl]))

# Calculating Test Data RMSE
rmse_googl_test = math.sqrt(mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl]))

# Calculating MAPE performance metrics
mape_googl_train = np.mean(np.abs((unscaled_y_train_googl - y_train_pred[:,googl])/unscaled_y_train_googl))*100

# Calculating Test Data MAPE
mape_googl_test = np.mean(np.abs((unscaled_y_test_googl - y_test_pred[:,googl])/unscaled_y_test_googl))*100

# Calculating R² performance metrics
r2_googl_train = r2_score(unscaled_y_train_googl, y_train_pred[:, googl])

# Calculating Test Data R²
r2_googl_test = r2_score(unscaled_y_test_googl, y_test_pred[:, googl])

print('Evaluation of GOOGL price','\nMAE Train:', mae_googl_train, '\nMAE Test:', mae_googl_test,
      '\nMSE Train:', mse_googl_train, '\nMSE Test:', mse_googl_test,
      '\nRMSE Train1:', rmse_googl_train, '\nRMSE Test1:', rmse_googl_test,
      '\nMAPE Train:', mape_googl_train, '\nMAPE Test:', mape_googl_test,
      '\nR² Train:', r2_googl_train, '\nR² Test:', r2_googl_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of BLL Price

# Calculating MAE performance metrics
mae_bll_train = mean_absolute_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MAE
mae_bll_test = mean_absolute_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating MSE performance metrics
mse_bll_train = mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MSE
mse_bll_test = mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating RMSE performance metrics
rmse_bll_train = math.sqrt(mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll]))

# Calculating Test Data RMSE
rmse_bll_test = math.sqrt(mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll]))

# Calculating MAPE performance metrics
mape_bll_train = np.mean(np.abs((unscaled_y_train_bll - y_train_pred[:,bll])/unscaled_y_train_bll))*100

# Calculating Test Data MAPE
mape_bll_test = np.mean(np.abs((unscaled_y_test_bll - y_test_pred[:,bll])/unscaled_y_test_bll))*100

# Calculating R² performance metrics
r2_bll_train = r2_score(unscaled_y_train_bll, y_train_pred[:, bll])

# Calculating Test Data R²
r2_bll_test = r2_score(unscaled_y_test_bll, y_test_pred[:, bll])

print('Evaluation of BLL price','\nMAE Train:', mae_bll_train, '\nMAE Test:', mae_bll_test,
      '\nMSE Train:', mse_bll_train, '\nMSE Test:', mse_bll_test,
      '\nRMSE Train1:', rmse_bll_train, '\nRMSE Test1:', rmse_bll_test,
      '\nMAPE Train:', mape_bll_train, '\nMAPE Test:', mape_bll_test,
      '\nR² Train:', r2_bll_train, '\nR² Test:', r2_bll_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of QCOM Price

# Calculating MAE performance metrics
mae_qcom_train = mean_absolute_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MAE
mae_qcom_test = mean_absolute_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating MSE performance metrics
mse_qcom_train = mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MSE
mse_qcom_test = mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating RMSE performance metrics
rmse_qcom_train = math.sqrt(mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom]))

# Calculating Test Data RMSE
rmse_qcom_test = math.sqrt(mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom]))

# Calculating MAPE performance metrics
mape_qcom_train = np.mean(np.abs((unscaled_y_train_qcom - y_train_pred[:,qcom])/unscaled_y_train_qcom))*100

# Calculating Test Data MAPE
mape_qcom_test = np.mean(np.abs((unscaled_y_test_qcom - y_test_pred[:,qcom])/unscaled_y_test_qcom))*100

# Calculating R² performance metrics
r2_qcom_train = r2_score(unscaled_y_train_qcom, y_train_pred[:, qcom])

# Calculating Test Data R²
r2_qcom_test = r2_score(unscaled_y_test_qcom, y_test_pred[:, qcom])

print('Evaluation of QCOM price','\nMAE Train:', mae_qcom_train, '\nMAE Test:', mae_qcom_test,
      '\nMSE Train:', mse_qcom_train, '\nMSE Test:', mse_qcom_test,
      '\nRMSE Train1:', rmse_qcom_train, '\nRMSE Test1:', rmse_qcom_test,
      '\nMAPE Train:', mape_qcom_train, '\nMAPE Test:', mape_qcom_test,
      '\nR² Train:', r2_qcom_train, '\nR² Test:', r2_qcom_test)

In [ ]:
# Save data to excel
df_pred_test_norm = pd.DataFrame(data=testpreds_arr)
df_pred_train_norm = pd.DataFrame(data=trainpreds_arr)
df_pred_test = pd.DataFrame(data=y_test_pred)
df_pred_train = pd.DataFrame(data=y_train_pred)

file_name15 = 'TestPredNormData4.xlsx'
file_name16 = 'TrainPredNormData4.xlsx'
file_name17 = 'TestPredData4.xlsx'
file_name18 = 'TrainPredData4.xlsx'

df_pred_test_norm.to_excel(file_name15)
df_pred_train_norm.to_excel(file_name16)
df_pred_test.to_excel(file_name17)
df_pred_train.to_excel(file_name18)

# **Model3**

In [ ]:
input_amzn = Input(shape=(num_days_used, num_features_amzn), name = 'input_amzn')
input_googl = Input(shape=(num_days_used, num_features_googl), name = 'input_googl')
input_bll = Input(shape=(num_days_used, num_features_bll), name = 'input_bll')
input_qcom = Input(shape=(num_days_used, num_features_qcom), name = 'input_qcom')

# LSTM branch for efficient initial processing
x1 = GRU(160, return_sequences=True, name='gru_amzn')(input_amzn)
x1 = Dropout(0.5)(x1)

# GRU branch to capture long-term dependencies in parallel
x1 = LSTM(160, return_sequences=True, name='lstm_amzn')(x1)
x1 = Dropout(0.5)(x1)

# LSTM branch for efficient initial processing
x2 = GRU(160, return_sequences=True, name='gru_googl')(input_googl)
x2 = Dropout(0.5)(x2)

# GRU branch to capture long-term dependencies in parallel
x2 = LSTM(160, return_sequences=True, name='lstm_googl')(x2)
x2 = Dropout(0.5)(x2)

# LSTM branch for efficient initial processing
x3 = GRU(160, return_sequences=True, name='gru_bll')(input_bll)
x3 = Dropout(0.5)(x3)

# GRU branch to capture long-term dependencies in parallel
x3 = LSTM(160, return_sequences=True, name='lstm_bll')(x3)
x3 = Dropout(0.5)(x3)

# LSTM branch for efficient initial processing
x4 = GRU(160, return_sequences=True, name='gru_qcom')(input_qcom)
x4 = Dropout(0.5)(x4)

# GRU branch to capture long-term dependencies in parallel
x4 = LSTM(160, return_sequences=True, name='lstm_qcom')(x4)
x4 = Dropout(0.5)(x4)

conc = concatenate([x1,x2,x3,x4])
conc = LSTM(160, return_sequences=True, name='lstm_conc')(conc)
conc = Dropout(0.5)(conc)

output1 = GRU(160, name='amzn_0')(conc)
output1 = Dense(1, name='amzn_final')(output1)

output2 = GRU(160, name='googl_0')(conc)
output2 = Dense(1, name='googl_final')(output2)

output3 = GRU(160, name='bll_0')(conc)
output3 = Dense(1, name='bll_final')(output3)

output4 = GRU(160, name='qcom_0')(conc)
output4 = Dense(1, name='qcom_final')(output4)


model5 = Model(inputs = [input_amzn, input_googl, input_bll, input_qcom], outputs = [output1, output2, output3, output4])

adam = Adam(learning_rate=0.001)

model5.compile(optimizer=adam, loss='mse')
model5.summary()

In [ ]:
# Displaying the structure of the final model
plot_model(model5, show_shapes=True)

In [ ]:
# Fitting Model
history = model5.fit(x=[X_train_amzn,X_train_googl,X_train_bll,X_train_qcom], y=[y_train_amzn,y_train_googl,y_train_bll,y_train_qcom], batch_size=32, epochs=30, validation_split=0.2)
evaluation = model5.evaluate([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom], [y_test_amzn,y_test_googl,y_test_bll,y_test_qcom])
print(evaluation)

In [ ]:
# Prediction data test
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model5.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
preds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_test_pred = preds_arr

amzn=0
googl=1
bll=2
qcom=3

plt.gcf().set_size_inches(22, 15, forward=True)
currentFig.set_facecolor('white')

real = plt.plot(y_test[:,:], label='real')
pred = plt.plot(y_test_pred[:,:], label='predicted')

plt.legend(['real amzn','real googl','real bll','real qcom','predict amzn','predic googl','predict bll','predict qcom'])
plt.xlabel('Days being predicted (units are arbitrary)', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close Price on the Test Set', fontsize=30)

plt.show()

In [ ]:
# Prediction data of each company
y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred= model5.predict([X_train_amzn,X_train_googl,X_train_bll,X_train_qcom])
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model5.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
trainpreds_arr = np.hstack((y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred))
testpreds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_train_pred = y_normaliser.inverse_transform(trainpreds_arr)
y_test_pred = y_normaliser.inverse_transform(testpreds_arr)

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_amzn'], label='real amzn price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,amzn], label='predicted train amzn', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,amzn], label='predicted test amzn', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close AMZN Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_googl'], label='real googl price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,googl], label='predicted train googl', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,googl], label='predicted test googl', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close GOOGL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_bll'], label='real bll price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,bll], label='predicted train bll', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,bll], label='predicted test bll', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close BLL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_qcom'], label='real qcom price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,qcom], label='predicted train qcom', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,qcom], label='predicted test qcom', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close QCOM Price on the Train and Test Set', fontsize=30)
plt.show()

In [ ]:
# Visualization Loss
loss = history.history['loss']
val_loss = history.history['val_loss']
amzn_loss = history.history['amzn_final_loss']
val_amzn_loss = history.history['val_amzn_final_loss']
googl_loss = history.history['googl_final_loss']
val_googl_loss = history.history['val_googl_final_loss']
bll_loss = history.history['bll_final_loss']
val_bll_loss = history.history['val_bll_final_loss']
qcom_loss = history.history['qcom_final_loss']
val_qcom_loss = history.history['val_qcom_final_loss']
epochs = range(1, len(loss) + 1)
plt.figure()

#Train and validation loss
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()
plt.show()

plt.plot(epochs, amzn_loss, 'b', label='Training loss')
plt.plot(epochs, val_amzn_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of AMZN')
plt.legend()
plt.show()

plt.plot(epochs, googl_loss, 'b', label='Training loss')
plt.plot(epochs, val_googl_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of GOOGL')
plt.legend()
plt.show()

plt.plot(epochs, bll_loss, 'b', label='Training loss')
plt.plot(epochs, val_bll_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of BLL')
plt.legend()
plt.show()

plt.plot(epochs, qcom_loss, 'b', label='Training loss')
plt.plot(epochs, val_qcom_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of QCOM')
plt.legend()
plt.show()

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of AMZN Price

# Calculating MAE performance metrics
mae_amzn_train = mean_absolute_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MAE
mae_amzn_test = mean_absolute_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating MSE performance metrics
mse_amzn_train = mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MSE
mse_amzn_test = mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating RMSE performance metrics
rmse_amzn_train = math.sqrt(mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn]))

# Calculating Test Data RMSE
rmse_amzn_test = math.sqrt(mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn]))

# Calculating MAPE performance metrics
mape_amzn_train = np.mean(np.abs((unscaled_y_train_amzn - y_train_pred[:,amzn])/unscaled_y_train_amzn))*100

# Calculating Test Data MAPE
mape_amzn_test = np.mean(np.abs((unscaled_y_test_amzn - y_test_pred[:,amzn])/unscaled_y_test_amzn))*100

# Calculating R² performance metrics
r2_amzn_train = r2_score(unscaled_y_train_amzn, y_train_pred[:, amzn])

# Calculating Test Data R²
r2_amzn_test = r2_score(unscaled_y_test_amzn, y_test_pred[:, amzn])

print('Evaluation of AMZN price','\nMAE Train:', mae_amzn_train, '\nMAE Test:', mae_amzn_test,
      '\nMSE Train:', mse_amzn_train, '\nMSE Test:', mse_amzn_test,
      '\nRMSE Train1:', rmse_amzn_train, '\nRMSE Test1:', rmse_amzn_test,
      '\nMAPE Train:', mape_amzn_train, '\nMAPE Test:', mape_amzn_test,
      '\nR² Train:', r2_amzn_train, '\nR² Test:', r2_amzn_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE, r2 of GOOGL Price

# Calculating MAE performance metrics
mae_googl_train = mean_absolute_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MAE
mae_googl_test = mean_absolute_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating MSE performance metrics
mse_googl_train = mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MSE
mse_googl_test = mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating RMSE performance metrics
rmse_googl_train = math.sqrt(mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl]))

# Calculating Test Data RMSE
rmse_googl_test = math.sqrt(mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl]))

# Calculating MAPE performance metrics
mape_googl_train = np.mean(np.abs((unscaled_y_train_googl - y_train_pred[:,googl])/unscaled_y_train_googl))*100

# Calculating Test Data MAPE
mape_googl_test = np.mean(np.abs((unscaled_y_test_googl - y_test_pred[:,googl])/unscaled_y_test_googl))*100

# Calculating R² performance metrics
r2_googl_train = r2_score(unscaled_y_train_googl, y_train_pred[:, googl])

# Calculating Test Data R²
r2_googl_test = r2_score(unscaled_y_test_googl, y_test_pred[:, googl])

print('Evaluation of GOOGL price','\nMAE Train:', mae_googl_train, '\nMAE Test:', mae_googl_test,
      '\nMSE Train:', mse_googl_train, '\nMSE Test:', mse_googl_test,
      '\nRMSE Train1:', rmse_googl_train, '\nRMSE Test1:', rmse_googl_test,
      '\nMAPE Train:', mape_googl_train, '\nMAPE Test:', mape_googl_test,
      '\nR² Train:', r2_googl_train, '\nR² Test:', r2_googl_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of BLL Price

# Calculating MAE performance metrics
mae_bll_train = mean_absolute_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MAE
mae_bll_test = mean_absolute_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating MSE performance metrics
mse_bll_train = mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MSE
mse_bll_test = mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating RMSE performance metrics
rmse_bll_train = math.sqrt(mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll]))

# Calculating Test Data RMSE
rmse_bll_test = math.sqrt(mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll]))

# Calculating MAPE performance metrics
mape_bll_train = np.mean(np.abs((unscaled_y_train_bll - y_train_pred[:,bll])/unscaled_y_train_bll))*100

# Calculating Test Data MAPE
mape_bll_test = np.mean(np.abs((unscaled_y_test_bll - y_test_pred[:,bll])/unscaled_y_test_bll))*100

# Calculating R² performance metrics
r2_bll_train = r2_score(unscaled_y_train_bll, y_train_pred[:, bll])

# Calculating Test Data R²
r2_bll_test = r2_score(unscaled_y_test_bll, y_test_pred[:, bll])

print('Evaluation of BLL price','\nMAE Train:', mae_bll_train, '\nMAE Test:', mae_bll_test,
      '\nMSE Train:', mse_bll_train, '\nMSE Test:', mse_bll_test,
      '\nRMSE Train1:', rmse_bll_train, '\nRMSE Test1:', rmse_bll_test,
      '\nMAPE Train:', mape_bll_train, '\nMAPE Test:', mape_bll_test,
      '\nR² Train:', r2_bll_train, '\nR² Test:', r2_bll_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of QCOM Price

# Calculating MAE performance metrics
mae_qcom_train = mean_absolute_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MAE
mae_qcom_test = mean_absolute_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating MSE performance metrics
mse_qcom_train = mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MSE
mse_qcom_test = mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating RMSE performance metrics
rmse_qcom_train = math.sqrt(mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom]))

# Calculating Test Data RMSE
rmse_qcom_test = math.sqrt(mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom]))

# Calculating MAPE performance metrics
mape_qcom_train = np.mean(np.abs((unscaled_y_train_qcom - y_train_pred[:,qcom])/unscaled_y_train_qcom))*100

# Calculating Test Data MAPE
mape_qcom_test = np.mean(np.abs((unscaled_y_test_qcom - y_test_pred[:,qcom])/unscaled_y_test_qcom))*100

# Calculating R² performance metrics
r2_qcom_train = r2_score(unscaled_y_train_qcom, y_train_pred[:, qcom])

# Calculating Test Data R²
r2_qcom_test = r2_score(unscaled_y_test_qcom, y_test_pred[:, qcom])

print('Evaluation of QCOM price','\nMAE Train:', mae_qcom_train, '\nMAE Test:', mae_qcom_test,
      '\nMSE Train:', mse_qcom_train, '\nMSE Test:', mse_qcom_test,
      '\nRMSE Train1:', rmse_qcom_train, '\nRMSE Test1:', rmse_qcom_test,
      '\nMAPE Train:', mape_qcom_train, '\nMAPE Test:', mape_qcom_test,
      '\nR² Train:', r2_qcom_train, '\nR² Test:', r2_qcom_test)

In [ ]:
# Save data to excel
df_pred_test_norm = pd.DataFrame(data=testpreds_arr)
df_pred_train_norm = pd.DataFrame(data=trainpreds_arr)
df_pred_test = pd.DataFrame(data=y_test_pred)
df_pred_train = pd.DataFrame(data=y_train_pred)

file_name19 = 'TestPredNormData5.xlsx'
file_name20 = 'TrainPredNormData5.xlsx'
file_name21 = 'TestPredData5.xlsx'
file_name22 = 'TrainPredData5.xlsx'

df_pred_test_norm.to_excel(file_name19)
df_pred_train_norm.to_excel(file_name20)
df_pred_test.to_excel(file_name21)
df_pred_train.to_excel(file_name22)

# **Model4**

In [ ]:
input_amzn = Input(shape=(num_days_used, num_features_amzn), name = 'input_amzn')
input_googl = Input(shape=(num_days_used, num_features_googl), name = 'input_googl')
input_bll = Input(shape=(num_days_used, num_features_bll), name = 'input_bll')
input_qcom = Input(shape=(num_days_used, num_features_qcom), name = 'input_qcom')

# LSTM branch for efficient initial processing
x1 = GRU(160, return_sequences=True, name='gru_amzn')(input_amzn)
x1 = Dropout(0.5)(x1)

# GRU branch to capture long-term dependencies in parallel
x1 = LSTM(160, return_sequences=True, name='lstm_amzn')(x1)
x1 = Dropout(0.5)(x1)

# LSTM branch for efficient initial processing
x2 = GRU(160, return_sequences=True, name='gru_googl')(input_googl)
x2 = Dropout(0.5)(x2)

# GRU branch to capture long-term dependencies in parallel
x2 = LSTM(160, return_sequences=True, name='lstm_googl')(x2)
x2 = Dropout(0.5)(x2)

# LSTM branch for efficient initial processing
x3 = GRU(160, return_sequences=True, name='gru_bll')(input_bll)
x3 = Dropout(0.5)(x3)

# GRU branch to capture long-term dependencies in parallel
x3 = LSTM(160, return_sequences=True, name='lstm_bll')(x3)
x3 = Dropout(0.5)(x3)

# LSTM branch for efficient initial processing
x4 = GRU(160, return_sequences=True, name='gru_qcom')(input_qcom)
x4 = Dropout(0.5)(x4)

# GRU branch to capture long-term dependencies in parallel
x4 = LSTM(160, return_sequences=True, name='lstm_qcom')(x4)
x4 = Dropout(0.5)(x4)

conc = concatenate([x1,x2,x3,x4])
conc = LSTM(160, return_sequences=True, name='lstm_conc1')(conc)
conc = LSTM(160, return_sequences=True, name='lstm_conc2')(conc)

output1 = GRU(160, name='amzn_0')(conc)
output1 = Dense(1, name='amzn_final')(output1)

output2 = GRU(160, name='googl_0')(conc)
output2 = Dense(1, name='googl_final')(output2)

output3 = GRU(160, name='bll_0')(conc)
output3 = Dense(1, name='bll_final')(output3)

output4 = GRU(160, name='qcom_0')(conc)
output4 = Dense(1, name='qcom_final')(output4)

model6 = Model(inputs = [input_amzn, input_googl, input_bll, input_qcom], outputs = [output1, output2, output3, output4])

adam = Adam(learning_rate=0.001)

model6.compile(optimizer=adam, loss='mse')
model6.summary()

In [ ]:
# Displaying the structure of the final model
plot_model(model6, show_shapes=True)

In [ ]:
# Fitting Model
history = model6.fit(x=[X_train_amzn,X_train_googl,X_train_bll,X_train_qcom], y=[y_train_amzn,y_train_googl,y_train_bll,y_train_qcom], batch_size=32, epochs=30, validation_split=0.2)
evaluation = model6.evaluate([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom], [y_test_amzn,y_test_googl,y_test_bll,y_test_qcom])
print(evaluation)

In [ ]:
# Prediction data test
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model6.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
preds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_test_pred = preds_arr

amzn=0
googl=1
bll=2
qcom=3

plt.gcf().set_size_inches(22, 15, forward=True)
currentFig.set_facecolor('white')

real = plt.plot(y_test[:,:], label='real')
pred = plt.plot(y_test_pred[:,:], label='predicted')

plt.legend(['real amzn','real googl','real bll','real qcom','predict amzn','predic googl','predict bll','predict qcom'])
plt.xlabel('Days being predicted (units are arbitrary)', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close Price on the Test Set', fontsize=30)

plt.show()

In [ ]:
# Prediction data of each company
y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred= model6.predict([X_train_amzn,X_train_googl,X_train_bll,X_train_qcom])
y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred= model6.predict([X_test_amzn,X_test_googl,X_test_bll,X_test_qcom])
trainpreds_arr = np.hstack((y_train_amzn_pred, y_train_googl_pred, y_train_bll_pred, y_train_qcom_pred))
testpreds_arr = np.hstack((y_test_amzn_pred, y_test_googl_pred, y_test_bll_pred, y_test_qcom_pred))
y_train_pred = y_normaliser.inverse_transform(trainpreds_arr)
y_test_pred = y_normaliser.inverse_transform(testpreds_arr)

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_amzn'], label='real amzn price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,amzn], label='predicted train amzn', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,amzn], label='predicted test amzn', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close AMZN Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_googl'], label='real googl price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,googl], label='predicted train googl', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,googl], label='predicted test googl', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close GOOGL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_bll'], label='real bll price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,bll], label='predicted train bll', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,bll], label='predicted test bll', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close BLL Price on the Train and Test Set', fontsize=30)
plt.show()

plt.gcf().set_size_inches(22, 15, forward=True)
# real values plotted
plt.plot(stock_df['Close_qcom'], label='real qcom price', color='g')

# predicted values plotted
plt.plot(dates_train, y_train_pred[:,qcom], label='predicted train qcom', color='lightgreen')
plt.plot(dates_test, y_test_pred[:,qcom], label='predicted test qcom', color='r', linestyle='dashed')

currentFig.set_facecolor('white')
plt.legend()
plt.xlabel('Year', fontsize=20)
plt.ylabel('Price (USD)', fontsize=20)
plt.title('Real and Predicted Close QCOM Price on the Train and Test Set', fontsize=30)
plt.show()

In [ ]:
# Visualization Loss
loss = history.history['loss']
val_loss = history.history['val_loss']
amzn_loss = history.history['amzn_final_loss']
val_amzn_loss = history.history['val_amzn_final_loss']
googl_loss = history.history['googl_final_loss']
val_googl_loss = history.history['val_googl_final_loss']
bll_loss = history.history['bll_final_loss']
val_bll_loss = history.history['val_bll_final_loss']
qcom_loss = history.history['qcom_final_loss']
val_qcom_loss = history.history['val_qcom_final_loss']
epochs = range(1, len(loss) + 1)
plt.figure()

#Train and validation loss
plt.plot(epochs, loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss')
plt.legend()
plt.show()

plt.plot(epochs, amzn_loss, 'b', label='Training loss')
plt.plot(epochs, val_amzn_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of AMZN')
plt.legend()
plt.show()

plt.plot(epochs, googl_loss, 'b', label='Training loss')
plt.plot(epochs, val_googl_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of GOOGL')
plt.legend()
plt.show()

plt.plot(epochs, bll_loss, 'b', label='Training loss')
plt.plot(epochs, val_bll_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of BLL')
plt.legend()
plt.show()

plt.plot(epochs, qcom_loss, 'b', label='Training loss')
plt.plot(epochs, val_qcom_loss, 'r', label='Validation loss')
plt.title('Training and Validation loss of QCOM')
plt.legend()
plt.show()

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of AMZN Price

# Calculating MAE performance metrics
mae_amzn_train = mean_absolute_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MAE
mae_amzn_test = mean_absolute_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating MSE performance metrics
mse_amzn_train = mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn])

# Calculating Test Data MSE
mse_amzn_test = mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn])

# Calculating RMSE performance metrics
rmse_amzn_train = math.sqrt(mean_squared_error(unscaled_y_train_amzn, y_train_pred[:,amzn]))

# Calculating Test Data RMSE
rmse_amzn_test = math.sqrt(mean_squared_error(unscaled_y_test_amzn, y_test_pred[:,amzn]))

# Calculating MAPE performance metrics
mape_amzn_train = np.mean(np.abs((unscaled_y_train_amzn - y_train_pred[:,amzn])/unscaled_y_train_amzn))*100

# Calculating Test Data MAPE
mape_amzn_test = np.mean(np.abs((unscaled_y_test_amzn - y_test_pred[:,amzn])/unscaled_y_test_amzn))*100

# Calculating R² performance metrics
r2_amzn_train = r2_score(unscaled_y_train_amzn, y_train_pred[:, amzn])

# Calculating Test Data R²
r2_amzn_test = r2_score(unscaled_y_test_amzn, y_test_pred[:, amzn])

print('Evaluation of AMZN price','\nMAE Train:', mae_amzn_train, '\nMAE Test:', mae_amzn_test,
      '\nMSE Train:', mse_amzn_train, '\nMSE Test:', mse_amzn_test,
      '\nRMSE Train1:', rmse_amzn_train, '\nRMSE Test1:', rmse_amzn_test,
      '\nMAPE Train:', mape_amzn_train, '\nMAPE Test:', mape_amzn_test,
      '\nR² Train:', r2_amzn_train, '\nR² Test:', r2_amzn_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE, r2 of GOOGL Price

# Calculating MAE performance metrics
mae_googl_train = mean_absolute_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MAE
mae_googl_test = mean_absolute_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating MSE performance metrics
mse_googl_train = mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl])

# Calculating Test Data MSE
mse_googl_test = mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl])

# Calculating RMSE performance metrics
rmse_googl_train = math.sqrt(mean_squared_error(unscaled_y_train_googl, y_train_pred[:,googl]))

# Calculating Test Data RMSE
rmse_googl_test = math.sqrt(mean_squared_error(unscaled_y_test_googl, y_test_pred[:,googl]))

# Calculating MAPE performance metrics
mape_googl_train = np.mean(np.abs((unscaled_y_train_googl - y_train_pred[:,googl])/unscaled_y_train_googl))*100

# Calculating Test Data MAPE
mape_googl_test = np.mean(np.abs((unscaled_y_test_googl - y_test_pred[:,googl])/unscaled_y_test_googl))*100

# Calculating R² performance metrics
r2_googl_train = r2_score(unscaled_y_train_googl, y_train_pred[:, googl])

# Calculating Test Data R²
r2_googl_test = r2_score(unscaled_y_test_googl, y_test_pred[:, googl])

print('Evaluation of GOOGL price','\nMAE Train:', mae_googl_train, '\nMAE Test:', mae_googl_test,
      '\nMSE Train:', mse_googl_train, '\nMSE Test:', mse_googl_test,
      '\nRMSE Train1:', rmse_googl_train, '\nRMSE Test1:', rmse_googl_test,
      '\nMAPE Train:', mape_googl_train, '\nMAPE Test:', mape_googl_test,
      '\nR² Train:', r2_googl_train, '\nR² Test:', r2_googl_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of BLL Price

# Calculating MAE performance metrics
mae_bll_train = mean_absolute_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MAE
mae_bll_test = mean_absolute_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating MSE performance metrics
mse_bll_train = mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll])

# Calculating Test Data MSE
mse_bll_test = mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll])

# Calculating RMSE performance metrics
rmse_bll_train = math.sqrt(mean_squared_error(unscaled_y_train_bll, y_train_pred[:,bll]))

# Calculating Test Data RMSE
rmse_bll_test = math.sqrt(mean_squared_error(unscaled_y_test_bll, y_test_pred[:,bll]))

# Calculating MAPE performance metrics
mape_bll_train = np.mean(np.abs((unscaled_y_train_bll - y_train_pred[:,bll])/unscaled_y_train_bll))*100

# Calculating Test Data MAPE
mape_bll_test = np.mean(np.abs((unscaled_y_test_bll - y_test_pred[:,bll])/unscaled_y_test_bll))*100

# Calculating R² performance metrics
r2_bll_train = r2_score(unscaled_y_train_bll, y_train_pred[:, bll])

# Calculating Test Data R²
r2_bll_test = r2_score(unscaled_y_test_bll, y_test_pred[:, bll])

print('Evaluation of BLL price','\nMAE Train:', mae_bll_train, '\nMAE Test:', mae_bll_test,
      '\nMSE Train:', mse_bll_train, '\nMSE Test:', mse_bll_test,
      '\nRMSE Train1:', rmse_bll_train, '\nRMSE Test1:', rmse_bll_test,
      '\nMAPE Train:', mape_bll_train, '\nMAPE Test:', mape_bll_test,
      '\nR² Train:', r2_bll_train, '\nR² Test:', r2_bll_test)

In [ ]:
#Evaluation MAE, MSE, RMSE, MAPE of QCOM Price

# Calculating MAE performance metrics
mae_qcom_train = mean_absolute_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MAE
mae_qcom_test = mean_absolute_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating MSE performance metrics
mse_qcom_train = mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom])

# Calculating Test Data MSE
mse_qcom_test = mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom])

# Calculating RMSE performance metrics
rmse_qcom_train = math.sqrt(mean_squared_error(unscaled_y_train_qcom, y_train_pred[:,qcom]))

# Calculating Test Data RMSE
rmse_qcom_test = math.sqrt(mean_squared_error(unscaled_y_test_qcom, y_test_pred[:,qcom]))

# Calculating MAPE performance metrics
mape_qcom_train = np.mean(np.abs((unscaled_y_train_qcom - y_train_pred[:,qcom])/unscaled_y_train_qcom))*100

# Calculating Test Data MAPE
mape_qcom_test = np.mean(np.abs((unscaled_y_test_qcom - y_test_pred[:,qcom])/unscaled_y_test_qcom))*100

# Calculating R² performance metrics
r2_qcom_train = r2_score(unscaled_y_train_qcom, y_train_pred[:, qcom])

# Calculating Test Data R²
r2_qcom_test = r2_score(unscaled_y_test_qcom, y_test_pred[:, qcom])

print('Evaluation of QCOM price','\nMAE Train:', mae_qcom_train, '\nMAE Test:', mae_qcom_test,
      '\nMSE Train:', mse_qcom_train, '\nMSE Test:', mse_qcom_test,
      '\nRMSE Train1:', rmse_qcom_train, '\nRMSE Test1:', rmse_qcom_test,
      '\nMAPE Train:', mape_qcom_train, '\nMAPE Test:', mape_qcom_test,
      '\nR² Train:', r2_qcom_train, '\nR² Test:', r2_qcom_test)

In [ ]:
# Save data to excel
df_pred_test_norm = pd.DataFrame(data=testpreds_arr)
df_pred_train_norm = pd.DataFrame(data=trainpreds_arr)
df_pred_test = pd.DataFrame(data=y_test_pred)
df_pred_train = pd.DataFrame(data=y_train_pred)

file_name23 = 'TestPredNormData6.xlsx'
file_name24 = 'TrainPredNormData6.xlsx'
file_name25 = 'TestPredData6.xlsx'
file_name26 = 'TrainPredData6.xlsx'

df_pred_test_norm.to_excel(file_name23)
df_pred_train_norm.to_excel(file_name24)
df_pred_test.to_excel(file_name25)
df_pred_train.to_excel(file_name26)

# **Accuracy MAPE (Mean Absolute Percentage Error), RMSPE (Roor Mean Square Percentage Error) and RMDPE (Root Mean Dimention Percentage Error) Hybrid GRU to LSTM**

In [ ]:
import os
os.listdir('/content/')

In [ ]:

with pd.ExcelWriter('/content/GRU_LSTM.xlsx') as writer:
    # Original data
    pd.read_excel('/content/RealData.xlsx').to_excel(writer, sheet_name='Data Asli', index=False)

    # Model 3 predictions
    pd.read_excel('/content/TrainPredData3.xlsx').to_excel(writer, sheet_name='Model3', startrow=0, index=False)
    pd.read_excel('/content/TestPredData3.xlsx').to_excel(writer, sheet_name='Model3', startrow=2403, index=False)

    # Model 4 predictions
    pd.read_excel('/content/TrainPredData4.xlsx').to_excel(writer, sheet_name='Model4', startrow=0, index=False)
    pd.read_excel('/content/TestPredData4.xlsx').to_excel(writer, sheet_name='Model4', startrow=2403, index=False)

    # Model 5 predictions
    pd.read_excel('/content/TrainPredData5.xlsx').to_excel(writer, sheet_name='Model5', startrow=0, index=False)
    pd.read_excel('/content/TestPredData5.xlsx').to_excel(writer, sheet_name='Model5', startrow=2403, index=False)

    # Model 6 predictions
    pd.read_excel('/content/TrainPredData6.xlsx').to_excel(writer, sheet_name='Model6', startrow=0, index=False)
    pd.read_excel('/content/TestPredData6.xlsx').to_excel(writer, sheet_name='Model6', startrow=2403, index=False)

In [ ]:


# access data excel from drive
path_gru = '/content/GRU_LSTM.xlsx'
df_gru = pd.ExcelFile(path_gru)

In [ ]:

# Read data excel
df0_gru = pd.read_excel(path_gru, 'Data Asli')
df3_gru = pd.read_excel(path_gru, 'Model3')
df4_gru = pd.read_excel(path_gru, 'Model4')
df5_gru = pd.read_excel(path_gru, 'Model5')
df6_gru = pd.read_excel(path_gru, 'Model6')

In [ ]:
df0_gru

In [ ]:
df4_gru

In [ ]:

#drop unused data column
df0_gru.drop(['Unnamed: 0'],axis=1,inplace=True)
df3_gru.drop(['Unnamed: 0'],axis=1,inplace=True)
df4_gru.drop(['Unnamed: 0'],axis=1,inplace=True)
df5_gru.drop(['Unnamed: 0'],axis=1,inplace=True)
df6_gru.drop(['Unnamed: 0'],axis=1,inplace=True)

In [ ]:
df0_gru

In [ ]:
df3_gru

In [ ]:

# # Get data test
# unscaled_data_gru = df0_gru.to_numpy()#[2443:][0:len(df0_gru)-2443, :]
# unscaled_df3_gru = df3_gru.to_numpy()#[2403:][0:len(df3_gru)-2403, :]
# unscaled_df4_gru = df4_gru.to_numpy()#[2403:][0:len(df4_gru)-2403, :]
# unscaled_df5_gru = df5_gru.to_numpy()#[2403:][0:len(df5_gru)-2403, :]
# unscaled_df6_gru = df6_gru.to_numpy()#[2403:][0:len(df6_gru)-2403, :]

In [ ]:

# # Get test data (last 603 rows of predictions)
test_len = len(df0_gru) - 2443   # same logic as real test set

unscaled_data_gru = df0_gru.to_numpy()[-test_len:, :]   # real values
unscaled_df3_gru  = df3_gru.to_numpy()[-test_len:, :]   # Model 3 predictions
unscaled_df4_gru  = df4_gru.to_numpy()[-test_len:, :]   # Model 4 predictions
unscaled_df5_gru  = df5_gru.to_numpy()[-test_len:, :]   # Model 5 predictions
unscaled_df6_gru  = df6_gru.to_numpy()[-test_len:, :]   # Model 6 predictions


In [ ]:
print(df3_gru.shape)
print(df3_gru.head())

In [ ]:
unscaled_data_gru.shape

In [ ]:

unscaled_df3_gru

In [ ]:
amzn = 0
googl = 1
bll = 2
qcom = 3

p = 40

# Calculate MAPE (Mean Absolute Percentage Error), RMSPE (Roor Mean Square Percentage Error) and RMDPE (Root Mean Dimention Percentage Error) for each company

# These calculations are done across the whole data set (Train + Test)

#AMZN
# MAPE
mape_amzn_model3_gru = np.mean(np.abs((unscaled_data_gru[:,amzn] - unscaled_df3_gru[:,amzn])/unscaled_data_gru[:,amzn]))*100
accuracy_mape_amzn_model3_gru = 100-mape_amzn_model3_gru
mape_amzn_model4_gru = np.mean(np.abs((unscaled_data_gru[:,amzn] - unscaled_df4_gru[:,amzn])/unscaled_data_gru[:,amzn]))*100
accuracy_mape_amzn_model4_gru = 100-mape_amzn_model4_gru
mape_amzn_model5_gru = np.mean(np.abs((unscaled_data_gru[:,amzn] - unscaled_df5_gru[:,amzn])/unscaled_data_gru[:,amzn]))*100
accuracy_mape_amzn_model5_gru = 100-mape_amzn_model5_gru
mape_amzn_model6_gru = np.mean(np.abs((unscaled_data_gru[:,amzn] - unscaled_df6_gru[:,amzn])/unscaled_data_gru[:,amzn]))*100
accuracy_mape_amzn_model6_gru = 100-mape_amzn_model6_gru

amzn_model3_gru = ((unscaled_df3_gru[:,amzn] - unscaled_data_gru[:,amzn])/unscaled_data_gru[:,amzn])
amzn_model4_gru = ((unscaled_df4_gru[:,amzn] - unscaled_data_gru[:,amzn])/unscaled_data_gru[:,amzn])
amzn_model5_gru = ((unscaled_df5_gru[:,amzn] - unscaled_data_gru[:,amzn])/unscaled_data_gru[:,amzn])
amzn_model6_gru = ((unscaled_df6_gru[:,amzn] - unscaled_data_gru[:,amzn])/unscaled_data_gru[:,amzn])

# RMSPE
rmspe_amzn_model3_gru = np.array([None]*601)
for i in range(601):
  rmspe_amzn_model3_gru[i] = math.pow(amzn_model3_gru[i],2)
rmspe_amzn_model3_gru = math.sqrt(np.mean(rmspe_amzn_model3_gru))*100
accuracy_rmspe_amzn_model3_gru = 100-rmspe_amzn_model3_gru
rmspe_amzn_model4_gru = np.array([None]*601)
for i in range(601):
  rmspe_amzn_model4_gru[i] = math.pow(amzn_model4_gru[i],2)
rmspe_amzn_model4_gru = math.sqrt(np.mean(rmspe_amzn_model4_gru))*100
accuracy_rmspe_amzn_model4_gru = 100-rmspe_amzn_model4_gru
rmspe_amzn_model5_gru = np.array([None]*601)
for i in range(601):
  rmspe_amzn_model5_gru[i] = math.pow(amzn_model5_gru[i],2)
rmspe_amzn_model5_gru = math.sqrt(np.mean(rmspe_amzn_model5_gru))*100
accuracy_rmspe_amzn_model5_gru = 100-rmspe_amzn_model5_gru
rmspe_amzn_model6_gru = np.array([None]*601)
for i in range(601):
  rmspe_amzn_model6_gru[i] = math.pow(amzn_model6_gru[i],2)
rmspe_amzn_model6_gru = math.sqrt(np.mean(rmspe_amzn_model6_gru))*100
accuracy_rmspe_amzn_model6_gru = 100-rmspe_amzn_model6_gru

# RMDPE
rmdpe_amzn_model3_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_amzn_model3_gru[i] = abs(amzn_model3_gru[i]) ** p
rmdpe_amzn_model3_gru = (np.mean(rmdpe_amzn_model3_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_amzn_model3_gru = 100 - rmdpe_amzn_model3_gru

rmdpe_amzn_model4_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_amzn_model4_gru[i] = abs(amzn_model4_gru[i]) ** p
rmdpe_amzn_model4_gru = (np.mean(rmdpe_amzn_model4_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_amzn_model4_gru = 100 - rmdpe_amzn_model4_gru

rmdpe_amzn_model5_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_amzn_model5_gru[i] = abs(amzn_model5_gru[i]) ** p
rmdpe_amzn_model5_gru = (np.mean(rmdpe_amzn_model5_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_amzn_model5_gru = 100 - rmdpe_amzn_model5_gru

rmdpe_amzn_model6_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_amzn_model6_gru[i] = abs(amzn_model6_gru[i]) ** p
rmdpe_amzn_model6_gru = (np.mean(rmdpe_amzn_model6_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_amzn_model6_gru = 100 - rmdpe_amzn_model6_gru

#GOOGL
# MAPE
mape_googl_model3_gru = np.mean(np.abs((unscaled_data_gru[:,googl] - unscaled_df3_gru[:,googl])/unscaled_data_gru[:,googl]))*100
accuracy_mape_googl_model3_gru = 100-mape_googl_model3_gru
mape_googl_model4_gru = np.mean(np.abs((unscaled_data_gru[:,googl] - unscaled_df4_gru[:,googl])/unscaled_data_gru[:,googl]))*100
accuracy_mape_googl_model4_gru = 100-mape_googl_model4_gru
mape_googl_model5_gru = np.mean(np.abs((unscaled_data_gru[:,googl] - unscaled_df5_gru[:,googl])/unscaled_data_gru[:,googl]))*100
accuracy_mape_googl_model5_gru = 100-mape_googl_model5_gru
mape_googl_model6_gru = np.mean(np.abs((unscaled_data_gru[:,googl] - unscaled_df6_gru[:,googl])/unscaled_data_gru[:,googl]))*100
accuracy_mape_googl_model6_gru = 100-mape_googl_model6_gru

googl_model3_gru = ((unscaled_df3_gru[:,googl] - unscaled_data_gru[:,googl])/unscaled_data_gru[:,googl])
googl_model4_gru = ((unscaled_df4_gru[:,googl] - unscaled_data_gru[:,googl])/unscaled_data_gru[:,googl])
googl_model5_gru = ((unscaled_df5_gru[:,googl] - unscaled_data_gru[:,googl])/unscaled_data_gru[:,googl])
googl_model6_gru = ((unscaled_df6_gru[:,googl] - unscaled_data_gru[:,googl])/unscaled_data_gru[:,googl])

# RMSPE
rmspe_googl_model3_gru = np.array([None]*601)
for i in range(601):
  rmspe_googl_model3_gru[i] = math.pow(googl_model3_gru[i],2)
rmspe_googl_model3_gru = math.sqrt(np.mean(rmspe_googl_model3_gru))*100
accuracy_rmspe_googl_model3_gru = 100-rmspe_googl_model3_gru
rmspe_googl_model4_gru = np.array([None]*601)
for i in range(601):
  rmspe_googl_model4_gru[i] = math.pow(googl_model4_gru[i],2)
rmspe_googl_model4_gru = math.sqrt(np.mean(rmspe_googl_model4_gru))*100
accuracy_rmspe_googl_model4_gru = 100-rmspe_googl_model4_gru
rmspe_googl_model5_gru = np.array([None]*601)
for i in range(601):
  rmspe_googl_model5_gru[i] = math.pow(googl_model5_gru[i],2)
rmspe_googl_model5_gru = math.sqrt(np.mean(rmspe_googl_model5_gru))*100
accuracy_rmspe_googl_model5_gru = 100-rmspe_googl_model5_gru
rmspe_googl_model6_gru = np.array([None]*601)
for i in range(601):
  rmspe_googl_model6_gru[i] = math.pow(googl_model6_gru[i],2)
rmspe_googl_model6_gru = math.sqrt(np.mean(rmspe_googl_model6_gru))*100
accuracy_rmspe_googl_model6_gru = 100-rmspe_googl_model6_gru

# RMDPE
rmdpe_googl_model3_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_googl_model3_gru[i] = abs(googl_model3_gru[i]) ** p
rmdpe_googl_model3_gru = (np.mean(rmdpe_googl_model3_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_googl_model3_gru = 100 - rmdpe_googl_model3_gru

rmdpe_googl_model4_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_googl_model4_gru[i] = abs(googl_model4_gru[i]) ** p
rmdpe_googl_model4_gru = (np.mean(rmdpe_googl_model4_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_googl_model4_gru = 100 - rmdpe_googl_model4_gru

rmdpe_googl_model5_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_googl_model5_gru[i] = abs(googl_model5_gru[i]) ** p
rmdpe_googl_model5_gru = (np.mean(rmdpe_googl_model5_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_googl_model5_gru = 100 - rmdpe_googl_model5_gru

rmdpe_googl_model6_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_googl_model6_gru[i] = abs(googl_model6_gru[i]) ** p
rmdpe_googl_model6_gru = (np.mean(rmdpe_googl_model6_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_googl_model6_gru = 100 - rmdpe_googl_model6_gru

#BLL
# MAPE
mape_bll_model3_gru = np.mean(np.abs((unscaled_data_gru[:,bll] - unscaled_df3_gru[:,bll])/unscaled_data_gru[:,bll]))*100
accuracy_mape_bll_model3_gru = 100-mape_bll_model3_gru
mape_bll_model4_gru = np.mean(np.abs((unscaled_data_gru[:,bll] - unscaled_df4_gru[:,bll])/unscaled_data_gru[:,bll]))*100
accuracy_mape_bll_model4_gru = 100-mape_bll_model4_gru
mape_bll_model5_gru = np.mean(np.abs((unscaled_data_gru[:,bll] - unscaled_df5_gru[:,bll])/unscaled_data_gru[:,bll]))*100
accuracy_mape_bll_model5_gru = 100-mape_bll_model5_gru
mape_bll_model6_gru = np.mean(np.abs((unscaled_data_gru[:,bll] - unscaled_df6_gru[:,bll])/unscaled_data_gru[:,bll]))*100
accuracy_mape_bll_model6_gru = 100-mape_bll_model6_gru

bll_model3_gru = ((unscaled_df3_gru[:,bll] - unscaled_data_gru[:,bll])/unscaled_data_gru[:,bll])
bll_model4_gru = ((unscaled_df4_gru[:,bll] - unscaled_data_gru[:,bll])/unscaled_data_gru[:,bll])
bll_model5_gru = ((unscaled_df5_gru[:,bll] - unscaled_data_gru[:,bll])/unscaled_data_gru[:,bll])
bll_model6_gru = ((unscaled_df6_gru[:,bll] - unscaled_data_gru[:,bll])/unscaled_data_gru[:,bll])

# RMSPE
rmspe_bll_model3_gru = np.array([None]*601)
for i in range(601):
  rmspe_bll_model3_gru[i] = math.pow(bll_model3_gru[i],2)
rmspe_bll_model3_gru = math.sqrt(np.mean(rmspe_bll_model3_gru))*100
accuracy_rmspe_bll_model3_gru = 100-rmspe_bll_model3_gru
rmspe_bll_model4_gru = np.array([None]*601)
for i in range(601):
  rmspe_bll_model4_gru[i] = math.pow(bll_model4_gru[i],2)
rmspe_bll_model4_gru = math.sqrt(np.mean(rmspe_bll_model4_gru))*100
accuracy_rmspe_bll_model4_gru = 100-rmspe_bll_model4_gru
rmspe_bll_model5_gru = np.array([None]*601)
for i in range(601):
  rmspe_bll_model5_gru[i] = math.pow(bll_model5_gru[i],2)
rmspe_bll_model5_gru = math.sqrt(np.mean(rmspe_bll_model5_gru))*100
accuracy_rmspe_bll_model5_gru = 100-rmspe_bll_model5_gru
rmspe_bll_model6_gru = np.array([None]*601)
for i in range(601):
  rmspe_bll_model6_gru[i] = math.pow(bll_model6_gru[i],2)
rmspe_bll_model6_gru = math.sqrt(np.mean(rmspe_bll_model6_gru))*100
accuracy_rmspe_bll_model6_gru = 100-rmspe_bll_model6_gru

# RMDPE
rmdpe_bll_model3_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_bll_model3_gru[i] = abs(bll_model3_gru[i]) ** p
rmdpe_bll_model3_gru = (np.mean(rmdpe_bll_model3_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_bll_model3_gru = 100 - rmdpe_bll_model3_gru

rmdpe_bll_model4_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_bll_model4_gru[i] = abs(bll_model4_gru[i]) ** p
rmdpe_bll_model4_gru = (np.mean(rmdpe_bll_model4_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_bll_model4_gru = 100 - rmdpe_bll_model4_gru

rmdpe_bll_model5_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_bll_model5_gru[i] = abs(bll_model5_gru[i]) ** p
rmdpe_bll_model5_gru = (np.mean(rmdpe_bll_model5_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_bll_model5_gru = 100 - rmdpe_bll_model5_gru

rmdpe_bll_model6_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_bll_model6_gru[i] = abs(bll_model6_gru[i]) ** p
rmdpe_bll_model6_gru = (np.mean(rmdpe_bll_model6_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_bll_model6_gru = 100 - rmdpe_bll_model6_gru

#QCOM
# MAPE
mape_qcom_model3_gru = np.mean(np.abs((unscaled_data_gru[:,qcom] - unscaled_df3_gru[:,qcom])/unscaled_data_gru[:,qcom]))*100
accuracy_mape_qcom_model3_gru = 100-mape_qcom_model3_gru
mape_qcom_model4_gru = np.mean(np.abs((unscaled_data_gru[:,qcom] - unscaled_df4_gru[:,qcom])/unscaled_data_gru[:,qcom]))*100
accuracy_mape_qcom_model4_gru = 100-mape_qcom_model4_gru
mape_qcom_model5_gru = np.mean(np.abs((unscaled_data_gru[:,qcom] - unscaled_df5_gru[:,qcom])/unscaled_data_gru[:,qcom]))*100
accuracy_mape_qcom_model5_gru = 100-mape_qcom_model5_gru
mape_qcom_model6_gru = np.mean(np.abs((unscaled_data_gru[:,qcom] - unscaled_df6_gru[:,qcom])/unscaled_data_gru[:,qcom]))*100
accuracy_mape_qcom_model6_gru = 100-mape_qcom_model6_gru

qcom_model3_gru = ((unscaled_df3_gru[:,qcom] - unscaled_data_gru[:,qcom])/unscaled_data_gru[:,qcom])
qcom_model4_gru = ((unscaled_df4_gru[:,qcom] - unscaled_data_gru[:,qcom])/unscaled_data_gru[:,qcom])
qcom_model5_gru = ((unscaled_df5_gru[:,qcom] - unscaled_data_gru[:,qcom])/unscaled_data_gru[:,qcom])
qcom_model6_gru = ((unscaled_df6_gru[:,qcom] - unscaled_data_gru[:,qcom])/unscaled_data_gru[:,qcom])

# RMSPE
rmspe_qcom_model3_gru = np.array([None]*601)
for i in range(601):
  rmspe_qcom_model3_gru[i] = math.pow(qcom_model3_gru[i],2)
rmspe_qcom_model3_gru = math.sqrt(np.mean(rmspe_qcom_model3_gru))*100
accuracy_rmspe_qcom_model3_gru = 100-rmspe_qcom_model3_gru
rmspe_qcom_model4_gru = np.array([None]*601)
for i in range(601):
  rmspe_qcom_model4_gru[i] = math.pow(qcom_model4_gru[i],2)
rmspe_qcom_model4_gru = math.sqrt(np.mean(rmspe_qcom_model4_gru))*100
accuracy_rmspe_qcom_model4_gru = 100-rmspe_qcom_model4_gru
rmspe_qcom_model5_gru = np.array([None]*601)
for i in range(601):
  rmspe_qcom_model5_gru[i] = math.pow(qcom_model5_gru[i],2)
rmspe_qcom_model5_gru = math.sqrt(np.mean(rmspe_qcom_model5_gru))*100
accuracy_rmspe_qcom_model5_gru = 100-rmspe_qcom_model5_gru
rmspe_qcom_model6_gru = np.array([None]*601)
for i in range(601):
  rmspe_qcom_model6_gru[i] = math.pow(qcom_model6_gru[i],2)
rmspe_qcom_model6_gru = math.sqrt(np.mean(rmspe_qcom_model6_gru))*100
accuracy_rmspe_qcom_model6_gru = 100-rmspe_qcom_model6_gru

# RMDPE
rmdpe_qcom_model3_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_qcom_model3_gru[i] = abs(qcom_model3_gru[i]) ** p
rmdpe_qcom_model3_gru = (np.mean(rmdpe_qcom_model3_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_qcom_model3_gru = 100 - rmdpe_qcom_model3_gru

rmdpe_qcom_model4_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_qcom_model4_gru[i] = abs(qcom_model4_gru[i]) ** p
rmdpe_qcom_model4_gru = (np.mean(rmdpe_qcom_model4_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_qcom_model4_gru = 100 - rmdpe_qcom_model4_gru

rmdpe_qcom_model5_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_qcom_model5_gru[i] = abs(qcom_model5_gru[i]) ** p
rmdpe_qcom_model5_gru = (np.mean(rmdpe_qcom_model5_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_qcom_model5_gru = 100 - rmdpe_qcom_model5_gru

rmdpe_qcom_model6_gru = np.array([None] * 601)
for i in range(601):
    rmdpe_qcom_model6_gru[i] = abs(qcom_model6_gru[i]) ** p
rmdpe_qcom_model6_gru = (np.mean(rmdpe_qcom_model6_gru)) ** (1.0 / p) * 100
accuracy_rmdpe_qcom_model6_gru = 100 - rmdpe_qcom_model6_gru

print('\n\nMODEL 1 GRU', '\nMAPE_model3_AMZN:', mape_amzn_model3_gru, '\nRMSPE_model3_AMZN:', rmspe_amzn_model3_gru, '\nRMDPE_model3_AMZN:', rmdpe_amzn_model3_gru,
      '\nAccuracy_MAPE_model3_AMZN:', accuracy_mape_amzn_model3_gru, '\nAccuracy_RMSPE_model3_AMZN:', accuracy_rmspe_amzn_model3_gru,
      '\nAccuracy_RMDPE_model3_AMZN:', accuracy_rmdpe_amzn_model3_gru,
      '\n\nMAPE_model3_GOOGL:', mape_googl_model3_gru, '\nRMSPE_model3_GOOGL:', rmspe_googl_model3_gru, '\nRMDPE_model3_GOOGL:', rmdpe_googl_model3_gru,
      '\nAccuracy_MAPE_model3_GOOGL:', accuracy_mape_googl_model3_gru, '\nAccuracy_RMSPE_model3_GOOGL:', accuracy_rmspe_googl_model3_gru,
      '\nAccuracy_RMDPE_model3_GOOGL:', accuracy_rmdpe_googl_model3_gru,
      '\n\nMAPE_model3_BLL:', mape_bll_model3_gru, '\nRMSPE_model3_BLL:', rmspe_bll_model3_gru, '\nRMDPE_model3_BLL:', rmdpe_bll_model3_gru,
      '\nAccuracy_MAPE_model3_BLL:', accuracy_mape_bll_model3_gru, '\nAccuracy_RMSPE_model3_BLL:', accuracy_rmspe_bll_model3_gru,
      '\nAccuracy_RMDPE_model3_BLL:', accuracy_rmdpe_bll_model3_gru,
      '\n\nMAPE_model3_QCOM:', mape_qcom_model3_gru, '\nRMSPE_model3_QCOM:', rmspe_qcom_model3_gru, '\nRMDPE_model3_QCOM:', rmdpe_qcom_model3_gru,
      '\nAccuracy_MAPE_model3_QCOM:', accuracy_mape_qcom_model3_gru, '\nAccuracy_RMSPE_model3_QCOM:', accuracy_rmspe_qcom_model3_gru,
      '\nAccuracy_RMDPE_model3_QCOM:', accuracy_rmdpe_qcom_model3_gru,
    '\n\nMODEL 2 GRU', '\nMAPE_model4_AMZN:', mape_amzn_model4_gru, '\nRMSPE_model4_AMZN:', rmspe_amzn_model4_gru, '\nRMDPE_model4_AMZN:', rmdpe_amzn_model4_gru,
      '\nAccuracy_MAPE_model4_AMZN:', accuracy_mape_amzn_model4_gru, '\nAccuracy_RMSPE_model4_AMZN:', accuracy_rmspe_amzn_model4_gru,
      '\nAccuracy_RMDPE_model4_AMZN:', accuracy_rmdpe_amzn_model4_gru,
      '\n\nMAPE_model4_GOOGL:', mape_googl_model4_gru, '\nRMSPE_model4_GOOGL:', rmspe_googl_model4_gru, '\nRMDPE_model4_GOOGL:', rmdpe_googl_model4_gru,
      '\nAccuracy_MAPE_model4_GOOGL:', accuracy_mape_googl_model4_gru, '\nAccuracy_RMSPE_model4_GOOGL:', accuracy_rmspe_googl_model4_gru,
      '\nAccuracy_RMDPE_model4_GOOGL:', accuracy_rmdpe_googl_model4_gru,
      '\n\nMAPE_model4_BLL:', mape_bll_model4_gru, '\nRMSPE_model4_BLL:', rmspe_bll_model4_gru, '\nRMDPE_model4_BLL:', rmdpe_bll_model4_gru,
      '\nAccuracy_MAPE_model4_BLL:', accuracy_mape_bll_model4_gru, '\nAccuracy_RMSPE_model4_BLL:', accuracy_rmspe_bll_model4_gru,
      '\nAccuracy_RMDPE_model4_BLL:', accuracy_rmdpe_bll_model4_gru,
      '\n\nMAPE_model4_QCOM:', mape_qcom_model4_gru, '\nRMSPE_model4_QCOM:', rmspe_qcom_model4_gru, '\nRMDPE_model4_QCOM:', rmdpe_qcom_model4_gru,
      '\nAccuracy_MAPE_model4_QCOM:', accuracy_mape_qcom_model4_gru, '\nAccuracy_RMSPE_model4_QCOM:', accuracy_rmspe_qcom_model4_gru,
      '\nAccuracy_RMDPE_model4_QCOM:', accuracy_rmdpe_qcom_model4_gru,
    '\n\nMODEL 3 GRU', '\nMAPE_model5_AMZN:', mape_amzn_model5_gru, '\nRMSPE_model5_AMZN:', rmspe_amzn_model5_gru, '\nRMDPE_model5_AMZN:', rmdpe_amzn_model5_gru,
      '\nAccuracy_MAPE_model5_AMZN:', accuracy_mape_amzn_model5_gru, '\nAccuracy_RMSPE_model5_AMZN:', accuracy_rmspe_amzn_model5_gru,
      '\nAccuracy_RMDPE_model5_AMZN:', accuracy_rmdpe_amzn_model5_gru,
      '\n\nMAPE_model5_GOOGL:', mape_googl_model5_gru, '\nRMSPE_model5_GOOGL:', rmspe_googl_model5_gru, '\nRMDPE_model5_GOOGL:', rmdpe_googl_model5_gru,
      '\nAccuracy_MAPE_model5_GOOGL:', accuracy_mape_googl_model5_gru, '\nAccuracy_RMSPE_model5_GOOGL:', accuracy_rmspe_googl_model5_gru,
      '\nAccuracy_RMDPE_model5_GOOGL:', accuracy_rmdpe_googl_model5_gru,
      '\n\nMAPE_model5_BLL:', mape_bll_model5_gru, '\nRMSPE_model5_BLL:', rmspe_bll_model5_gru, '\nRMDPE_model5_BLL:', rmdpe_bll_model5_gru,
      '\nAccuracy_MAPE_model5_BLL:', accuracy_mape_bll_model5_gru, '\nAccuracy_RMSPE_model5_BLL:', accuracy_rmspe_bll_model5_gru,
      '\nAccuracy_RMDPE_model5_BLL:', accuracy_rmdpe_bll_model5_gru,
      '\n\nMAPE_model5_QCOM:', mape_qcom_model5_gru, '\nRMSPE_model5_QCOM:', rmspe_qcom_model5_gru, '\nRMDPE_model5_QCOM:', rmdpe_qcom_model5_gru,
      '\nAccuracy_MAPE_model5_QCOM:', accuracy_mape_qcom_model5_gru, '\nAccuracy_RMSPE_model5_QCOM:', accuracy_rmspe_qcom_model5_gru,
      '\nAccuracy_RMDPE_model5_QCOM:', accuracy_rmdpe_qcom_model5_gru,
    '\n\nMODEL 4 GRU', '\nMAPE_model6_AMZN:', mape_amzn_model6_gru, '\nRMSPE_model6_AMZN:', rmspe_amzn_model6_gru, '\nRMDPE_model6_AMZN:', rmdpe_amzn_model6_gru,
      '\nAccuracy_MAPE_model6_AMZN:', accuracy_mape_amzn_model6_gru, '\nAccuracy_RMSPE_model6_AMZN:', accuracy_rmspe_amzn_model6_gru,
      '\nAccuracy_RMDPE_model6_AMZN:', accuracy_rmdpe_amzn_model6_gru,
      '\n\nMAPE_model6_GOOGL:', mape_googl_model6_gru, '\nRMSPE_model6_GOOGL:', rmspe_googl_model6_gru, '\nRMDPE_model6_GOOGL:', rmdpe_googl_model6_gru,
      '\nAccuracy_MAPE_model6_GOOGL:', accuracy_mape_googl_model6_gru, '\nAccuracy_RMSPE_model6_GOOGL:', accuracy_rmspe_googl_model6_gru,
      '\nAccuracy_RMDPE_model6_GOOGL:', accuracy_rmdpe_googl_model6_gru,
      '\n\nMAPE_model6_BLL:', mape_bll_model6_gru, '\nRMSPE_model6_BLL:', rmspe_bll_model6_gru, '\nRMDPE_model6_BLL:', rmdpe_bll_model6_gru,
      '\nAccuracy_MAPE_model6_BLL:', accuracy_mape_bll_model6_gru, '\nAccuracy_RMSPE_model6_BLL:', accuracy_rmspe_bll_model6_gru,
      '\nAccuracy_RMDPE_model6_BLL:', accuracy_rmdpe_bll_model6_gru,
      '\n\nMAPE_model6_QCOM:', mape_qcom_model6_gru, '\nRMSPE_model6_QCOM:', rmspe_qcom_model6_gru, '\nRMDPE_model6_QCOM:', rmdpe_qcom_model6_gru,
      '\nAccuracy_MAPE_model6_QCOM:', accuracy_mape_qcom_model6_gru, '\nAccuracy_RMSPE_model6_QCOM:', accuracy_rmspe_qcom_model6_gru,
      '\nAccuracy_RMDPE_model6_QCOM:', accuracy_rmdpe_qcom_model6_gru)

In [ ]:
# Visualization accuracy model

#AMZN
amzn_gru_model3 = [accuracy_mape_amzn_model3_gru,accuracy_rmspe_amzn_model3_gru,accuracy_rmdpe_amzn_model3_gru]
amzn_gru_model4 = [accuracy_mape_amzn_model4_gru,accuracy_rmspe_amzn_model4_gru,accuracy_rmdpe_amzn_model4_gru]
amzn_gru_model5 = [accuracy_mape_amzn_model5_gru,accuracy_rmspe_amzn_model5_gru,accuracy_rmdpe_amzn_model5_gru]
amzn_gru_model6 = [accuracy_mape_amzn_model6_gru,accuracy_rmspe_amzn_model6_gru,accuracy_rmdpe_amzn_model6_gru]

j = 1
colors = ['r','g','b']

plt.bar(['MAPE','RMSPE','RMDPE'], amzn_gru_model3, width= 0.5, color=colors)
for i in range(len(amzn_gru_model3)):
    plt.annotate('%.2f'%amzn_gru_model3[i], (i, amzn_gru_model3[i] + j),horizontalalignment='center')
plt.ylabel('AMZN GRU_LSTM Model 3')
plt.title('AMZN GRU_LSTM Model 3')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], amzn_gru_model4, width= 0.5, color=colors)
for i in range(len(amzn_gru_model4)):
    plt.annotate('%.2f'%amzn_gru_model4[i], (i, amzn_gru_model4[i] + j),horizontalalignment='center')
plt.ylabel('AMZN GRU_LSTM Model 4')
plt.title('AMZN GRU_LSTM Model 4')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], amzn_gru_model5, width= 0.5, color=colors)
for i in range(len(amzn_gru_model5)):
    plt.annotate('%.2f'%amzn_gru_model5[i], (i, amzn_gru_model5[i] + j),horizontalalignment='center')
plt.ylabel('AMZN GRU_LSTM Model 5')
plt.title('AMZN GRU_LSTM Model 5')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], amzn_gru_model6, width= 0.5, color=colors)
for i in range(len(amzn_gru_model6)):
    plt.annotate('%.2f'%amzn_gru_model6[i], (i, amzn_gru_model6[i] + j),horizontalalignment='center')
plt.ylabel('AMZN GRU_LSTM Model 6')
plt.title('AMZN GRU_LSTM Model 6')
plt.show()

#GOOGL
googl_gru_model3 = [accuracy_mape_googl_model3_gru,accuracy_rmspe_googl_model3_gru,accuracy_rmdpe_googl_model3_gru]
googl_gru_model4 = [accuracy_mape_googl_model4_gru,accuracy_rmspe_googl_model4_gru,accuracy_rmdpe_googl_model4_gru]
googl_gru_model5 = [accuracy_mape_googl_model5_gru,accuracy_rmspe_googl_model5_gru,accuracy_rmdpe_googl_model5_gru]
googl_gru_model6 = [accuracy_mape_googl_model6_gru,accuracy_rmspe_googl_model6_gru,accuracy_rmdpe_googl_model6_gru]

j = 1
plt.bar(['MAPE','RMSPE','RMDPE'], googl_gru_model3, width= 0.5, color=colors)
for i in range(len(googl_gru_model3)):
    plt.annotate('%.2f'%googl_gru_model3[i], (i, googl_gru_model3[i] + j),horizontalalignment='center')
plt.ylabel('GOOGL GRU_LSTM Model 3')
plt.title('GOOGL GRU_LSTM Model 3')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], googl_gru_model4, width= 0.5, color=colors)
for i in range(len(googl_gru_model4)):
    plt.annotate('%.2f'%googl_gru_model4[i], (i, googl_gru_model4[i] + j),horizontalalignment='center')
plt.ylabel('GOOGL GRU_LSTM Model 4')
plt.title('GOOGL GRU_LSTM Model 4')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], googl_gru_model5, width= 0.5, color=colors)
for i in range(len(googl_gru_model5)):
    plt.annotate('%.2f'%googl_gru_model5[i], (i, googl_gru_model5[i] + j),horizontalalignment='center')
plt.ylabel('GOOGL GRU_LSTM Model 5')
plt.title('GOOGL GRU_LSTM Model 5')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], googl_gru_model6, width= 0.5, color=colors)
for i in range(len(googl_gru_model6)):
    plt.annotate('%.2f'%googl_gru_model6[i], (i, googl_gru_model6[i] + j),horizontalalignment='center')
plt.ylabel('GOOGL GRU_LSTM Model 6')
plt.title('GOOGL GRU_LSTM Model 6')
plt.show()

#BLL
bll_gru_model3 = [accuracy_mape_bll_model3_gru,accuracy_rmspe_bll_model3_gru,accuracy_rmdpe_bll_model3_gru]
bll_gru_model4 = [accuracy_mape_bll_model4_gru,accuracy_rmspe_bll_model4_gru,accuracy_rmdpe_bll_model4_gru]
bll_gru_model5 = [accuracy_mape_bll_model5_gru,accuracy_rmspe_bll_model5_gru,accuracy_rmdpe_bll_model5_gru]
bll_gru_model6 = [accuracy_mape_bll_model6_gru,accuracy_rmspe_bll_model6_gru,accuracy_rmdpe_bll_model6_gru]

j = 1
plt.bar(['MAPE','RMSPE','RMDPE'], bll_gru_model3, width= 0.5, color=colors)
for i in range(len(bll_gru_model3)):
    plt.annotate('%.2f'%bll_gru_model3[i], (i, bll_gru_model3[i] + j),horizontalalignment='center')
plt.ylabel('BLL GRU_LSTM Model 3')
plt.title('BLL GRU_LSTM Model 3')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], bll_gru_model4, width= 0.5, color=colors)
for i in range(len(bll_gru_model4)):
    plt.annotate('%.2f'%bll_gru_model4[i], (i, bll_gru_model4[i] + j),horizontalalignment='center')
plt.ylabel('BLL GRU_LSTM Model 4')
plt.title('BLL GRU_LSTM Model 4')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], bll_gru_model5, width= 0.5, color=colors)
for i in range(len(bll_gru_model5)):
    plt.annotate('%.2f'%bll_gru_model5[i], (i, bll_gru_model5[i] + j),horizontalalignment='center')
plt.ylabel('BLL GRU_LSTM Model 5')
plt.title('BLL GRU_LSTM Model 5')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], bll_gru_model6, width= 0.5, color=colors)
for i in range(len(bll_gru_model6)):
    plt.annotate('%.2f'%bll_gru_model6[i], (i, bll_gru_model6[i] + j),horizontalalignment='center')
plt.ylabel('BLL GRU_LSTM Model 6')
plt.title('BLL GRU_LSTM Model 6')
plt.show()

#QCOM
qcom_gru_model3 = [accuracy_mape_qcom_model3_gru,accuracy_rmspe_qcom_model3_gru,accuracy_rmdpe_qcom_model3_gru]
qcom_gru_model4 = [accuracy_mape_qcom_model4_gru,accuracy_rmspe_qcom_model4_gru,accuracy_rmdpe_qcom_model4_gru]
qcom_gru_model5 = [accuracy_mape_qcom_model5_gru,accuracy_rmspe_qcom_model5_gru,accuracy_rmdpe_qcom_model5_gru]
qcom_gru_model6 = [accuracy_mape_qcom_model6_gru,accuracy_rmspe_qcom_model6_gru,accuracy_rmdpe_qcom_model6_gru]

j = 1
plt.bar(['MAPE','RMSPE','RMDPE'], qcom_gru_model3, width= 0.5, color=colors)
for i in range(len(qcom_gru_model3)):
    plt.annotate('%.2f'%qcom_gru_model3[i], (i, qcom_gru_model3[i] + j),horizontalalignment='center')
plt.ylabel('QCOM GRU_LSTM Model 3')
plt.title('QCOM GRU_LSTM Model 3')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], qcom_gru_model4, width= 0.5, color=colors)
for i in range(len(qcom_gru_model4)):
    plt.annotate('%.2f'%qcom_gru_model4[i], (i, qcom_gru_model4[i] + j),horizontalalignment='center')
plt.ylabel('QCOM GRU_LSTM Model 4')
plt.title('QCOM GRU_LSTM Model 4')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], qcom_gru_model5, width= 0.5, color=colors)
for i in range(len(qcom_gru_model5)):
    plt.annotate('%.2f'%qcom_gru_model5[i], (i, qcom_gru_model5[i] + j),horizontalalignment='center')
plt.ylabel('QCOM GRU_LSTM Model 5')
plt.title('QCOM GRU_LSTM Model 5')
plt.show()

plt.bar(['MAPE','RMSPE','RMDPE'], qcom_gru_model6, width= 0.5, color=colors)
for i in range(len(qcom_gru_model6)):
    plt.annotate('%.2f'%qcom_gru_model6[i], (i, qcom_gru_model6[i] + j),horizontalalignment='center')
plt.ylabel('QCOM GRU_LSTM Model 6')
plt.title('QCOM GRU_LSTM Model 6')
plt.show()